# RegDet V1.1 — DAILY-FIT DIRECTION ("learn slow, apply fast")

Implements `DAILY_FIT_PROTOCOL.md` exactly. That protocol, and the four
predictions in section 2, were frozen before any number here existed.

**The change.** Fit the DIRECTION model on **daily** bars with the longest
available history, project the resulting direction state **causally** onto the 2h
bars, and leave the **intensity** (H vs L) axis on 2h, completely untouched.
Direction is slow, weak and needs many regimes to estimate; intensity is fast,
strong and needs fine resolution. Fit each where its information lives.

**Three arms, one knob at a time.**

| arm | direction fitted on | features |
|---|---|---|
| 1 BASELINE | 2h, ~2y | the existing 9 |
| 2 DAILY-9FEAT | daily, long history | the same 9 |
| 3 DAILY-3FEAT | daily, long history | decollinearized: one momentum, one vol, one dispersion |

Arm 2 isolates the **data** change. Arm 3 adds the **feature** change.

**The one thing that can silently invalidate all of this** is the daily→2h
projection. At a 2h bar on calendar day *d*, the direction state must come from
the last **fully closed** daily bar — day *d−1* or earlier. Section 4 enforces
that with a per-bar assert *and* a truncation probe that re-runs the entire
pipeline on a cut series and demands bit-identical labels. A failure there is a
hard stop, not something to work around.

`S`, `L`, `W` and guards `G1`–`G4` are **reused verbatim** from
`STABILITY_LAG_PROTOCOL.md` / `stability_lag.ipynb`, so rows here are directly
comparable with rows there. `BAR_DIR_WEIGHT = 0.0` throughout, asserted.

In [ ]:
%pip install -q hmmlearn yfinance

## 1. Engine, inlined verbatim

Inlined out of `build_master_notebook_v2.py` rather than imported, because a
standalone `.ipynb` on Kaggle cannot import a local `.py` next to it. Source:
`build_master_notebook_v2.py cell 3 (constants/imports/CONFIGS), cell 5 (load_2h/_synth), cell 7 (feature + labeling engine), cell 9 (regime-background plot helpers)`. Not one line is modified.

In [ ]:
# ==========================================================================
# CONSTANTS + IMPORTS -- INLINED VERBATIM from build_master_notebook_v2.py
# (master notebook code cell 3). Unmodified.
# ==========================================================================
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt          # default backend -> inline figures
import matplotlib.patches as mpatches
import matplotlib.dates as mdates        # date2num for the batched regime bands
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# ---------------------------------------------------------------------------
# Walk-forward harness config (matches capacity_ladder.py / the head-to-head)
# ---------------------------------------------------------------------------
HMM_ITER       = 2000          # EM iteration cap (fits converge in ~75-240 iters, <1s each)
N_FOLDS        = 4             # anchored walk-forward test blocks
MIN_TRAIN_FRAC = 0.50          # first fold trains on >= this fraction of bars
BARS_PER_DAY   = 3             # 2h bars per NSE session
LOOKBACK_SCALE = 1.0           # feature-window multiplier (1.0 = production windows)

# ---------------------------------------------------------------------------
# LABELING KNOBS — verbatim from REGDET_CONFIG in regdet_v11.py.
# These drive the direction + intensity + chop-filter scheme used EVERYWHERE in
# this notebook (walk-forward signal, regime overlay, evaluation suite).
# ---------------------------------------------------------------------------
CONF_L         = 0.50   # low-confidence override: if the WINNING direction bucket's
                        # aggregated probability mass is below this, the bar is forced
                        # to SIDEWAYS. Higher = more SIDEWAYS, fewer directional calls.
Z_HI           = 0.5    # trend-MAGNITUDE gate: |trend_z| must reach this for a bull/bear
                        # bar to escalate to H_BULL/H_BEAR. trend_z is TREND_FEATURE
                        # standardized against a mu/sd baseline FROZEN on the fit window
                        # (causal — never recomputed from bars the model has not seen).
                        # Lower = H fires more often (and flickers more).
EFF_HI         = 0.35   # trend-EFFICIENCY gate = THE CHOP FILTER. Efficiency is
                        # |net move| / |total path walked| over EFF_WIN bars: ~1.0 in a
                        # clean trend, ~0.0 in a range that keeps doubling back. Magnitude
                        # alone cannot separate a real trend from a swing inside a bracket
                        # (both show big momentum); this can. Requiring BOTH gates keeps
                        # range-bound swings at L_BULL/L_BEAR. Raise = stricter (less H).
EFF_WIN        = 9      # bars over which efficiency is measured (~ TREND_FEATURE horizon)
TREND_FEATURE  = 'mom_3d'   # the feature whose z-score grades trend magnitude

# ---------------------------------------------------------------------------
# LABELING FIXES 1-3 (see Section 7H for the measured A/B). Each is behind its
# own switch, and each switch has a value that reproduces the OLD behaviour
# bit-for-bit, so their effects can be separated and audited.
# ---------------------------------------------------------------------------

# FIX 1 -- DIRECTION SCORER: features that are excluded from the composite
# BULLISHNESS score. vol_2h and vol_expansion are MAGNITUDE measures: they say
# how big moves are, not which way they point. Feeding them into a DIRECTION
# score is a category error -- with the old weights the risk block (vol_2h,
# vol_expansion, vix_chg, drawdown; total 4.0) outweighed the directional block
# (ret_2h, mom_1d/3d/5d, dist_ma; total 3.2), so a violent RALLY (high vol, high
# vol expansion, still-elevated drawdown) scored BEARISH while price ripped up.
# These two features REMAIN HMM INPUT FEATURES -- they are genuinely informative
# about state -- they simply stop voting on direction. drawdown and vix_chg keep
# their -1.0 direction weight: both are defensibly signed (a deeper drawdown and
# a rising VIX really are bearish, not merely "big").
# DIRECTION_EXCLUDE = () reproduces the old scorer exactly.
DIRECTION_EXCLUDE = ('vol_2h', 'vol_expansion')

# FIX 2 -- LABEL HYSTERESIS (causal confirmation delay): a candidate new
# DIRECTION must persist for CONFIRM_BARS consecutive bars before the emitted
# label is allowed to flip; until then the previous emitted direction is held.
# This is a DELAY, not a smoother: bar t's emitted label is a function of bars
# <= t only. (An earlier version of this project shipped a "smoother" that
# decided whether to erase a run by inspecting the run's full REALIZED length --
# that is look-ahead and was removed. Nothing of that shape is reintroduced here;
# the causality probe in 7.0 re-proves it under hysteresis.)
# CONFIRM_BARS = 1 reproduces the old behaviour exactly.
CONFIRM_BARS   = 2

# FIX 3 -- GATE HYSTERESIS: the H escalation gates become enter/exit BANDS.
# Escalate L -> H when |z| >= Z_HI AND eff >= EFF_HI (unchanged), but only
# de-escalate H -> L when |z| < Z_HI_EXIT OR eff < EFF_HI_EXIT. Without this,
# bars sitting near a single threshold flicker H->L->H->L every bar.
# Z_HI_EXIT = Z_HI and EFF_HI_EXIT = EFF_HI reproduces the old behaviour exactly.
Z_HI_EXIT      = 0.35
EFF_HI_EXIT    = 0.25

# FIX 4 -- PER-BAR DIRECTION (this is the architectural one).
#
# THE FLAW IT ADDRESSES. Until now DIRECTION was a per-STATE property: the HMM
# assigns bar t a distribution over states, each STATE is bucketed bear/side/bull
# ONCE from the rank of its composite mean profile, and the bar inherits its
# state's direction. But an HMM state is directionally MIXED. Volatility is
# direction-agnostic, so the model reliably learns a "violent" state that
# contains BOTH sharp selloffs AND sharp rallies. ONE bucket label for that state
# cannot be right for every bar in it, no matter how the bucket is chosen.
#
# Real-data evidence (2874 2h Nifty bars) on the 366 causal high-vol RALLY bars
# (top vol_2h quartile AND trailing mom_3d > 0):
#   baseline          46.2% bear / 7.4% sideways / 46.4% bull
#   + FIX 1 (no vol)  11.2% bear / 42.3% sideways / 46.4% bull
# The red became GREY, not green -- bull did not move at all -- and SIDEWAYS then
# showed the HIGHEST forward return of any label (+0.717% at 15 bars vs H_BULL
# +0.026%), which is exactly what "a bullish population got parked in SIDEWAYS"
# looks like. FIX 1 removed a wrong vote; it could not add a right one, because
# the vote is cast once per state and not once per bar.
#
# THE FIX. Add a CAUSAL PER-BAR directional score from the SIGNED features only
# (ret_2h, mom_1d, mom_3d, mom_5d, dist_ma -- reusing FEATURE_SIGN/FEATURE_MAG),
# each standardized against a mu/sd baseline FROZEN on the fit window exactly the
# way trend_z is, then map it to three per-bar direction masses through a softmax
# and BLEND those with the state-level masses:
#
#     mass = (1 - BAR_DIR_WEIGHT) * state_mass + BAR_DIR_WEIGHT * bar_mass
#
# A convex combination of two points on the 3-simplex is on the 3-simplex, so
# bull+side+bear == 1 still holds bar by bar and every downstream mechanism --
# the prob_* partition, tactical_regime_confidence, the CONF_L override, the
# intensity gates, the CONFIRM_BARS hysteresis -- is untouched. The HMM keeps
# supplying market character, persistence and the confidence signal; only the
# DIRECTION ATTRIBUTION moves from per-state to per-bar.
#
# vol_2h / vol_expansion are structurally excluded from this score: they are
# magnitude, not direction. That is asserted, not merely intended.
#
# BAR_DIR_WEIGHT = 0.0 reproduces the pre-fix behaviour BIT-FOR-BIT -- the blend
# degenerates to 1.0*state_mass + 0.0*bar_mass, which is exact in IEEE754 for
# non-negative masses. Section 7H asserts that equality rather than assuming it.
# BAR_DIR_WEIGHT = 1.0 decides direction purely per bar (the HMM then contributes
# only character/persistence, not direction). Section 7H sweeps 0/0.25/0.5/0.75/1.
#
# ADOPTED VALUE 0.0 -- FIX 4 IS RETAINED AS A SWITCH BUT SET OFF.
#
# It was 0.75 (and before that 0.5). It is now 0.0. The machinery, the sweep in
# 7H-vi and the overlay in 7H-vii all stay; only the shipped weight moved.
#
# WHY IT WAS TURNED OFF. Scored across three real-data runs, the benefit fix 4 was
# added for -- more bull on high-volatility rally bars -- did NOT reproduce:
# +19.7 pp once, +1.4 pp the second time, and on the third the rally gain came
# from fixes 1-2 with fix 4 already OFF. The COST reproduced every time: it broke
# the direction-level forward-return ordering. A controlled A/B on the SAME data
# and the SAME fit, changing only w:
#
#     metric                        w = 0.75        w = 0.0
#     direction ordering 3/9/15     BROKEN/HOLDS/BROKEN   HOLDS/HOLDS/HOLDS
#     H_BULL fwd_3 HAC t            1.93 WARN       2.06 PASS
#     L_BULL fwd_9 HAC t            1.04 FAIL       2.10 PASS
#     BULL vs BEAR d @ fwd_3        0.006           0.118
#     H_BULL vs H_BEAR d @ fwd_3    0.139 WARN      0.214 PASS
#     strategy total return         +17.37%         +37.74%
#     strategy Sharpe               0.69            1.26
#
# w = 0 is where this project first produced HAC t > 2 on anything.
#
# The dead-zone note that justified 0.75 over 0.5 is still true and still the
# reason NOT to ship an intermediate value if fix 4 is ever switched back on: the
# filtered state posterior saturates near one-hot (confidence ~0.999 on most
# bars), so a convex blend cannot move the argmax until w > ~0.57. The choice is
# effectively between 0.0 (off) and >= ~0.75 (on); 0.25/0.5 are nominally on and
# behaviourally almost off, which is the worst of both.
#
# At w = 0.0 `dir_feats` is STILL passed everywhere it was before. The blend
# degenerates to 1.0*state_mass + 0.0*bar_mass -- exact in IEEE754 -- so labels
# are bit-identical to the no-dir_feats path (asserted in 7H), while
# `bar_dir_score` stays populated as a diagnostic column and the causality probe
# in Section 7.0 keeps testing it.
BAR_DIR_WEIGHT   = 0.0
BAR_DIR_FEATURES = ('ret_2h', 'mom_1d', 'mom_3d', 'mom_5d', 'dist_ma')
BAR_DIR_TAU      = 1.0   # softmax temperature on the per-bar z. The score is
                         # re-standardized on the fit window, so tau is in units
                         # of fit-window sd: |z| ~ 0.5*tau is where the leading
                         # direction's per-bar mass crosses 0.5. Lower tau =
                         # more decisive (more extreme) per-bar masses.

TRAIN_FRACTION = 0.70   # anchored fit fraction used by the PRODUCTION-style single fit
                        # in Section 7 (the engine's own TRAIN_FRACTION). The walk-forward
                        # in Section 5 uses MIN_TRAIN_FRAC/N_FOLDS fold edges instead.

# ---------------------------------------------------------------------------
# SEED ENSEMBLE — the identifiability fix (see Section 5d).
#
# hmm.GaussianHMM is fit by EM, a LOCAL optimizer. With one fixed seed the fit
# is not identified: refitting the same bars with different seeds lands in
# different local optima (train log-likelihood spread of ~1000 nats across 8
# seeds on the full-cov config) which segment the data differently. Multi-restart
# EM keeping the best log-likelihood does NOT fix it -- it collapses the LL
# spread but the surviving optima still disagree on the segmentation.
#
# What works instead: fit K models with K different seeds and average the
# DIRECTION-BUCKET PROBABILITY MASSES (bull / side / bear). Raw HMM state indices
# are arbitrary and permute freely between fits, so they cannot be averaged --
# but direction masses are permutation-INVARIANT semantic quantities, so they
# can. Each model's 3 masses sum to 1, so their average does too, and the
# labeling scheme downstream is bit-for-bit the same; only the SOURCE of the
# masses changes.
#
# ENSEMBLE_K = 1 reproduces the old single-fit behaviour exactly.
#
# Cost scales linearly in K (K fits per training slice). K=6 was the size the
# offline study measured (direction-call agreement between independent pools
# 81.9% single -> 91.9% at K=6), and K=6 is now the ADOPTED value: it is the size
# the stability study actually measured, and the cost is linear. The earlier K=4
# compromise existed only because Section 5d re-ran the ENTIRE walk-forward in
# both arms; 5d is OFF by default now (RUN_SEED_STABILITY=False below), so the
# runtime argument for K=4 no longer applies.
ENSEMBLE_K     = 6
BASE_SEED      = 42     # ensemble seeds are BASE_SEED + 0..K-1 (deterministic,
                        # so every run of this notebook is reproducible)

# ---------------------------------------------------------------------------
# RUNTIME KNOBS. These change ONLY how fast the notebook runs, never what it
# computes. Both have a value that reproduces the original code path exactly.
# ---------------------------------------------------------------------------

# N_JOBS -- ensemble fit parallelism. DEFAULT 1 (serial), and that default is
# deliberate. Read this before changing it.
#
# The K members of a seed ensemble ARE independent and each IS fully determined
# by its own random_state, so at the level of the algorithm, fitting them
# concurrently cannot change anything. It does anyway, for a reason that has
# nothing to do with this notebook's logic: MEASURED on this environment,
# `GaussianHMM.fit` is not bit-reproducible across OpenBLAS thread counts. The
# same seed on the same rows gives model parameters differing by ~1.5e-11
# between a 1-thread and a 4-thread BLAS, because threaded reductions sum in a
# different order and ~100-170 EM iterations amplify the last-bit difference.
# joblib's loky backend pins each worker to ONE inner thread (correctly -- it is
# avoiding oversubscription), so a parallel fit lands on the 1-thread arithmetic
# while the serial fit here lands on the multi-thread arithmetic.
#
# Measured, on 4 fits of 1500x9 at N=5 full-cov:
#     serial, default threads          3.19s   <- what this notebook does
#     serial, BLAS pinned to 1 thread  2.88s   params differ by 1.5e-11
#     parallel, 1 inner thread         2.48s   BIT-IDENTICAL to the line above
#     parallel, 4 inner threads       33.64s   10x SLOWER (oversubscription)
#
# So parallelism is available but only at 1 inner thread, and that arm is
# bit-identical to serial-at-1-thread -- NOT to serial-at-default-threads. The
# available speedup is ~1.3x on the fits, and the price is moving every model
# parameter in the 11th decimal. In a notebook where a 3-bar data perturbation
# has already flipped the config winner, that is a bad trade, so it is NOT the
# default. N_JOBS = 1 reproduces the historical numbers exactly.
#
# If you set N_JOBS != 1 you are choosing a different (equally valid, not more
# accurate) floating-point path, and the headline numbers may move slightly.
# Section 7.0 asserts and REPORTS this rather than hiding it.
N_JOBS         = 1

# Section 5d is a ONE-TIME IDENTIFIABILITY DIAGNOSTIC, not part of the pipeline:
# it re-runs the ENTIRE walk-forward 36 times (3 configs x 6 seeds x 2 arms) to
# ask whether the seed ensemble stabilised the config ranking. That question has
# been answered, and NOTHING downstream reads any variable it defines -- so on a
# normal run it is ~2/3 of the total wall time spent re-confirming a settled
# result. Default OFF. Set True to re-run it (e.g. after changing ENSEMBLE_K,
# N_STATES, the features or the folds -- any of which reopens the question).
RUN_SEED_STABILITY = False

CONFIDENCE_THRESHOLD_H  = 0.70   # chart reference line only
CONFIDENCE_THRESHOLD_L  = CONF_L # the actual SIDEWAYS override
HMM_PROB_DROP_THRESHOLD = 0.20   # confidence drop -> transition warning
VIX_SPIKE_THRESHOLD     = 0.25   # bar-over-bar VIX jump -> transition warning

REGIME_LABELS = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']
REGIME_COLORS = {
    'H_BULL':   '#006400',
    'L_BULL':   '#90EE90',
    'SIDEWAYS': '#808080',
    'L_BEAR':   '#FFB6C1',
    'H_BEAR':   '#8B0000',
}

# Feature names follow regdet_v11.py exactly (ret_2h / vol_2h, not ret / vol).
FEATURE_COLS = ['ret_2h', 'mom_1d', 'mom_3d', 'mom_5d',
                'vol_2h', 'vol_expansion', 'vix_chg', 'drawdown', 'dist_ma']

# 5-feature primary subset from the feature-selection analysis (corr clustering
# + PCA + VIF): one representative per information family.
FEATURES_LEAN = ['ret_2h', 'mom_3d', 'vol_2h', 'vol_expansion', 'dist_ma']

BASE_WIN = dict(MOM_1D=1*BARS_PER_DAY, MOM_3D=3*BARS_PER_DAY, MOM_5D=5*BARS_PER_DAY,
                VOL_WIN=10, VOL_FAST=5, VOL_SLOW=20, SWING_WIN=20)

# Per-feature sign/magnitude weights for the subset-agnostic bullishness scorer.
#   raw (DIRECTION_EXCLUDE = ()):
#     score = ret_2h + 0.4*(m1+m3+m5) - vol_2h - vol_expansion - vix_chg - drawdown + dist_ma
#   with FIX 1 (DIRECTION_EXCLUDE = ('vol_2h','vol_expansion')) the two magnitude
#   terms drop out and the score becomes purely directional:
#     score = ret_2h + 0.4*(m1+m3+m5) - vix_chg - drawdown + dist_ma
# The exclusion is applied as a WEIGHT OF ZERO in `direction_weight` below, so it
# stays subset-agnostic: excluding a feature the subset does not contain is a
# no-op, and the HMM's own feature matrix is untouched.
FEATURE_SIGN = {
    'ret_2h': 1.0, 'mom_1d': 1.0, 'mom_3d': 1.0, 'mom_5d': 1.0, 'dist_ma': 1.0,
    'vol_2h': -1.0, 'vol_expansion': -1.0, 'vix_chg': -1.0, 'drawdown': -1.0,
}
FEATURE_MAG = {'mom_1d': 0.4, 'mom_3d': 0.4, 'mom_5d': 0.4}   # else 1.0

# ---------------------------------------------------------------------------
# FIX 5 -- INTENSITY_MODE: the SCALE-FREE H/L intensity gate.
#
# THE FLAW. H_BULL / H_BEAR escalate on a trend-MAGNITUDE gate
#     trend_z = (mom_3d - mu_fit) / sd_fit          |trend_z| >= Z_HI = 0.5
# with mu_fit / sd_fit frozen on the training window. Freezing them is correct
# for CAUSALITY -- but it makes the THRESHOLD meaningless. A constant threshold
# is only interpretable on a scale-free quantity, and trend_z is scale-free only
# if sd_fit happens to equal the CURRENT dispersion of mom_3d. Volatility
# clusters, so it never does: the gate's aggressiveness is governed by the ratio
# sd_fit / sd_now, which is an artefact of where the training cut was placed and
# has no economic meaning.
#
# Real-data evidence (the user's own run, same bars, same config -- ONLY the fit
# window differs):
#     fit 50%:  H_BULL 22.9%  L_BULL 16.2%  SIDEWAYS 36.0%  L_BEAR  8.3%  H_BEAR 16.5%
#     fit 70%:  H_BULL 21.8%  L_BULL 31.0%  SIDEWAYS 10.6%  L_BEAR 17.3%  H_BEAR 19.3%
# SIDEWAYS is 3.4x larger in one than the other. These are effectively two
# different detectors produced by an arbitrary backtest cut.
#
# THE FIX. trend_t = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the t-statistic of
# the 9-bar move. mom_3d is the 9-bar sum of log returns; vol_2h is the trailing
# per-bar return sd. The ratio is DIMENSIONLESS and CONTEMPORANEOUS, and needs no
# fit-window baseline at all -- the fix REMOVES a fit-window dependence rather
# than relocating it. Both inputs are trailing rolling windows at bar t, so it is
# fully causal.
#
# A REJECTED alternative, measured and discarded: a trailing 250-bar quantile of
# |mom_3d|. It was WORSE than the status quo (H-firing spread 44.7 pp vs 35.8 pp)
# because a trailing window is a LAGGING scale estimate -- when volatility drops
# the window is full of stale high-vol bars and the H-rate collapsed to 11%.
#
# THE THRESHOLD. |trend_t| >= 0.5 would fire on ~68% of bars, so the threshold is
# not hand-picked either: it is derived as a TARGET OCCUPANCY from the FIT WINDOW
# ONLY (causal; computed once from bars <= n_fit, never per bar).
#     H_TARGET_RATE = 0.25  -> enter threshold = the (1 - 0.25) quantile of
#                              |trend_t| over the leading n_fit bars
#     H_EXIT_SLACK  = 0.10  -> exit  threshold = the (1 - 0.35) quantile, i.e. a
#                              LOOSER bar, preserving FIX 3's enter/exit band
# The derived thresholds are PRINTED on every run (Section 7.0).
#
# INTENSITY_MODE = 'frozen_z' reproduces today's behaviour BIT-FOR-BIT and is
# asserted to do so in Section 7H-viii against a frozen verbatim copy of the
# pre-change `label_bars`.
INTENSITY_MODE = 'vol_norm'    # 'frozen_z' = pre-change | 'vol_norm' = adopted
MOM_3D_BARS    = BASE_WIN['MOM_3D']    # 9 -- the horizon mom_3d integrates over
H_TARGET_RATE  = 0.25   # target FIT-WINDOW occupancy of the H gate
H_EXIT_SLACK   = 0.10   # exit threshold sits at (H_TARGET_RATE + this) occupancy

# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE: how an HMM state maps to bear / side / bull.
#
# 'rank' (DEFAULT, ADOPTED) -- today's hard rank buckets: sort the N states by
# composite bullishness, bottom 2 bear, middle 1 side, top 2 bull. Rank-based so
# the side bucket is guaranteed non-empty (a sign+deadzone rule can empty it and
# silently make SIDEWAYS unreachable -- that bug was shipped once and reverted).
#
# 'soft' (IMPLEMENTED, SWITCHABLE, *NOT* ADOPTED) -- replace the step function of
# the RANK with a smooth function of the VALUE: two sigmoids on the cross-state
# standardized composite score give each state a (bull, side, bear) weight row,
# and the bar's masses become `probs @ W` instead of a hard column sum.
#
# WHY IT IS NOT ADOPTED, stated exactly. Soft bucketing improves stability AT THE
# SOURCE -- the state->direction map stops flipping wholesale when one state's
# score crosses another's, measured 3.2x more stable -- but it makes the EMITTED
# SIDEWAYS/BEAR occupancy gaps WORSE. The reason is the standardization: with
# only N=5 states the composite scores are standardized by the sd of those same 5
# numbers, so a single outlier state inflates the sd and drags every other
# state's z toward zero, which washes the whole map toward SIDEWAYS by a
# different amount in each fit. Until that standardization is fixed (a robust
# scale, or a scale that does not depend on the state count), 'soft' is NOT
# RECOMMENDED and 'rank' remains the default.
DIRECTION_MODE = 'rank'   # 'rank' = adopted | 'soft' = implemented, not recommended
DIR_TAU   = 0.6    # soft: crossover WIDTH of the bull/bear sigmoids (cross-state sd units)
DIR_C     = 0.5    # soft: crossover CENTRE -- how far from the cross-state mean a
                   #       state must sit before it counts as directional
DIR_SCALE = 'sd'   # soft: cross-state scale, 'sd' or 'mad' (robust). This is the
                   #       knob named in the paragraph above; 'mad' is the obvious
                   #       first thing to try when fixing the standardization.

# ---------------------------------------------------------------------------
# ESCALATION_DURING_HOLD -- what the intensity gate may do while the DIRECTION
# is being held by CONFIRM_BARS.
#
# THE COUPLING. FIX 2 holds the emitted DIRECTION for CONFIRM_BARS bars while a
# candidate flip confirms. The intensity gate is graded FRESH at every bar on a
# SEPARATE clock (its own enter/exit state machine). So on the bars where
# dir_raw != dir_emit -- measured at ~5-11% of bars -- the notebook emits a
# direction that the current evidence no longer supports, while the gate is free
# to escalate that stale direction to H. That is maximum conviction emitted at
# maximum uncertainty.
#
#   'allow'            -- the PRE-CHANGE behaviour. Escalation is independent of
#                         whether the direction is contested. Retained as the
#                         bit-for-bit off switch (asserted in 7H-viii).
#   'block'  (DEFAULT) -- no NEW escalation to H while dir_raw != dir_emit. An
#                         already-running H escalation is still held under the
#                         exit band; only fresh ENTERs are suppressed. A blocked
#                         escalation leaves the gate state at 0, so a LATER bar
#                         cannot "hold" an H run it never entered -- the
#                         suppression propagates forward past the contested bar,
#                         which is what keeps the ENTER-band invariant intact.
#   'demote'           -- as 'block', and additionally force an existing H down to
#                         L while contested. The gate run ENDS, so re-escalation
#                         after the contest resolves must clear the full ENTER
#                         band again.
#
# WHY 'block' IS THE DEFAULT -- and, precisely, what is NOT established.
#
# ESTABLISHED (contested-H prototype):
#   * Contested-H underperforms confirmed-H in 18/18 measured cells (2 fit cuts x
#     3 horizons x {bull, bear, pooled}). Every one of the 18 is negative.
#   * MECHANICAL CORROBORATION, independently verified: contested-H bars carry
#     trend_efficiency 0.365 / 0.445 versus 0.597 / 0.603 for confirmed-H -- i.e.
#     contested-H bars are 1.35-1.64x CHOPPIER. That is exactly the condition this
#     system is supposed to take SMALLER size in, and 'allow' prints MAXIMUM size
#     there.
#   * The cost of switching is negligible and structurally safe: only 6-7 bars
#     change (0.39-0.45%), EVERY change is an H -> L demotion, and NO bar changes
#     DIRECTION. 'block' therefore cannot introduce a new failure mode -- it can
#     only reduce conviction.
#   * Persistence slightly IMPROVES under 'block' (runs 293 -> 291), whereas
#     'demote' fragments runs (-> 311). Hence 'block', not 'demote'.
#
# NOT ESTABLISHED -- read this before quoting the above as a return result:
#   * NO single return comparison clears |t| >= 2 under BOTH the HAC and the n_eff
#     corrections. There are only 16-20 contested-H bars in the sample. That is a
#     POWER limitation, not evidence of no effect -- but it means the 18/18 is a
#     consistent DIRECTION, not a demonstrated return gain.
#   * The justification for defaulting this ON is therefore: consistent sign +
#     the efficiency evidence + the asymmetry of costs (a wrongly-suppressed H
#     costs a little upside; a wrongly-emitted H costs full size into chop).
#     It is NOT "contested-H loses money, significantly". Do not overstate it.
#
# CAUSALITY. Both dir_raw and dir_emit are computable from bars <= t (dir_raw is
# a per-bar argmax of causal masses; dir_emit is confirm_delay's left-to-right
# scan), so the contested mask is causal, and the suppression is applied inside
# the same single left-to-right pass the gate already used. This is PROVED by a
# prefix-truncation probe in Section 7.0 under all three settings, not asserted.
ESCALATION_DURING_HOLD = 'block'   # 'block' (adopted) | 'allow' (pre-change) | 'demote'

# Forward-return horizons used for EVALUATION ONLY (hindsight; never a label input).
FWD_HORIZONS = [3, 9, 15]        # ~1 day, ~3 days, ~5 days of 2h bars

# ---------------------------------------------------------------------------
# The three configs under test: V1.0 (prod baseline) vs Candidate A (lean-cov)
# vs Candidate B (lean-feat). All at N=5, CONF_L=0.5. This head-to-head is about
# MODEL CAPACITY (covariance type / feature subset) -- the labeling scheme below
# is identical for all three, so the comparison isolates capacity.
# ---------------------------------------------------------------------------
CONFIGS = [
    dict(name='V1.0 (prod)',  N=5, cov='full', features=FEATURE_COLS),
    dict(name='A: lean-cov',  N=5, cov='diag', features=FEATURE_COLS),
    dict(name='B: lean-feat', N=5, cov='diag', features=FEATURES_LEAN),
]

# ---------------------------------------------------------------------------
# THE ADOPTED CONFIG -- set EXPLICITLY, not inherited from the head-to-head.
#
# The head-to-head (Section 5) still runs in full and still reports its winner.
# But that ranking is KNOWN UNSTABLE: it is decided on worst-fold Sharpe, a
# single noisy number over 4 folds, and a 3-bar perturbation of the input series
# has already been observed to flip it. Letting the whole evaluation suite follow
# whichever config happened to win means the notebook can silently evaluate a
# different detector on two runs of the same code.
#
# So the evaluated config is PINNED here. Sections 5c and 7 both use it, which is
# also what makes those two overlays comparable: they then differ ONLY in the fit
# window (see the re-role note in 5c / 7A), not in model capacity.
#
# If the head-to-head winner differs from this, that DISAGREEMENT IS REPORTED
# loudly in Sections 5b, 7.0 and 8a rather than silently resolved either way.
ADOPTED_CONFIG_NAME = 'A: lean-cov'    # N=5, cov='diag', all 9 features
assert any(c['name'] == ADOPTED_CONFIG_NAME for c in CONFIGS), \
    f'ADOPTED_CONFIG_NAME {ADOPTED_CONFIG_NAME!r} is not one of the configs under test'

# ---- knob sanity (each of these has bitten this project at least once) ------
assert INTENSITY_MODE in ('frozen_z', 'vol_norm'), 'unknown INTENSITY_MODE'
assert DIRECTION_MODE in ('rank', 'soft'), 'unknown DIRECTION_MODE'
assert ESCALATION_DURING_HOLD in ('allow', 'block', 'demote'), 'unknown ESCALATION_DURING_HOLD'
assert MOM_3D_BARS == BASE_WIN['MOM_3D'] == 9, 'mom_3d is expected to integrate 9 bars'
assert 0.0 < H_TARGET_RATE < 1.0 and H_EXIT_SLACK >= 0.0
assert DIR_TAU > 0.0 and DIR_C >= 0.0 and DIR_SCALE in ('sd', 'mad')
assert not (set(BAR_DIR_FEATURES) & {'vol_2h', 'vol_expansion'}), \
    'a MAGNITUDE feature leaked into the per-bar DIRECTION score'

print(f"Labeling: direction+intensity  CONF_L={CONF_L}  Z_HI={Z_HI}  "
      f"EFF_HI={EFF_HI}  EFF_WIN={EFF_WIN}  TREND_FEATURE={TREND_FEATURE}")
print(f"Fixes   : [1] DIRECTION_EXCLUDE={DIRECTION_EXCLUDE or '() -> old scorer'}  "
      f"[2] CONFIRM_BARS={CONFIRM_BARS}"
      + ('  (=1 -> old behaviour)' if CONFIRM_BARS == 1 else '')
      + f"  [3] gate bands enter(|z|>={Z_HI}, eff>={EFF_HI}) "
        f"exit(|z|<{Z_HI_EXIT}, eff<{EFF_HI_EXIT})"
      + ('  (exit==enter -> old behaviour)'
         if (Z_HI_EXIT == Z_HI and EFF_HI_EXIT == EFF_HI) else ''))
print(f"          [4] BAR_DIR_WEIGHT={BAR_DIR_WEIGHT} (per-bar direction blend; "
      f"tau={BAR_DIR_TAU}, features={list(BAR_DIR_FEATURES)})")
if BAR_DIR_WEIGHT == 0.0:
    print("              FIX 4 IS RETAINED AS A SWITCH BUT SET OFF. Direction is "
          "100% per-STATE (HMM).")
    print("              WHY: it broke the direction-level forward-return ordering, "
          "and the rally")
    print("              benefit it was added for did not reproduce across runs "
          "(+19.7pp, then +1.4pp,")
    print("              then a run where the gain came from fixes 1-2 with fix 4 "
          "already off).")
    print("              NOT deleted: the blend, the 7H-vi sweep and the 7H-vii "
          "overlay all still run,")
    print("              and bar_dir_score is still computed (diagnostic only -- the "
          "blend is an exact")
    print("              arithmetic no-op at w=0.0, asserted bit-for-bit in 7H).")
else:
    print(f"              FIX 4 IS ON. NOTE: w={BAR_DIR_WEIGHT} is NOT the shipped "
          "default (0.0); it broke")
    print("              the direction-level forward-return ordering on real data.")
print(f"Fitting : SEED ENSEMBLE  ENSEMBLE_K={ENSEMBLE_K}  BASE_SEED={BASE_SEED}  "
      f"seeds={[BASE_SEED + i for i in range(ENSEMBLE_K)]}"
      + ('   (K=1 -> old single-fit behaviour)' if ENSEMBLE_K == 1 else ''))
print(f"          [5] INTENSITY_MODE={INTENSITY_MODE!r}"
      + ("  (scale-free t-stat gate; thresholds derived from the FIT WINDOW at "
         f"target occupancy {H_TARGET_RATE:.2f} enter / {H_TARGET_RATE + H_EXIT_SLACK:.2f} exit)"
         if INTENSITY_MODE == 'vol_norm'
         else f"  (pre-change frozen mu/sd z-score vs constants Z_HI={Z_HI}/Z_HI_EXIT={Z_HI_EXIT})"))
print(f"          [6] DIRECTION_MODE={DIRECTION_MODE!r}"
      + ("  (hard rank buckets -- adopted)" if DIRECTION_MODE == 'rank'
         else f"  (SOFT weights tau={DIR_TAU} c={DIR_C} scale={DIR_SCALE!r}"
              "  -- NOT RECOMMENDED, see the note in the config cell)"))
print(f"          [7] ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}"
      + ("  (=allow -> PRE-CHANGE behaviour: the gate may escalate a held/contested direction)"
         if ESCALATION_DURING_HOLD == 'allow'
         else "  (no NEW H escalation while dir_raw != dir_emit"
              + ("; existing H also demoted)" if ESCALATION_DURING_HOLD == 'demote' else ")")))
print(f"Harness : HMM_ITER={HMM_ITER}  N_FOLDS={N_FOLDS}  MIN_TRAIN_FRAC={MIN_TRAIN_FRAC}  "
      f"configs={[c['name'] for c in CONFIGS]}")
print(f"Adopted : evaluated config is PINNED to {ADOPTED_CONFIG_NAME!r} (Sections 5c and 7); "
      f"the head-to-head still runs and its winner is reported and compared in 5b / 7.0 / 8a")
print(f"Runtime : N_JOBS={N_JOBS} (ensemble fits" + ('  serial' if N_JOBS == 1 else ' in parallel')
      + f")  RUN_SEED_STABILITY={RUN_SEED_STABILITY}"
      + ('  (5d diagnostic SKIPPED -- see 5d)' if not RUN_SEED_STABILITY else '')
      + '   [speed only; neither knob can change a result]')

In [ ]:
# ==========================================================================
# DATA LOADER -- INLINED VERBATIM (master code cell 5).
# yfinance 60m -> 2h resample, with a synthetic fallback. Unmodified.
# ==========================================================================
def load_2h():
    """yfinance 60m->2h, else synthetic. Returns (nifty, vix, is_synth)."""
    try:
        import yfinance as yf

        def h2(tk):
            raw = yf.download(tk, interval='60m', period='730d',
                              auto_adjust=True, progress=False)
            if raw is None or len(raw) == 0:
                return None
            c = raw['Close'].squeeze().dropna()
            idx = pd.to_datetime(c.index)
            c.index = idx.tz_localize(None) if idx.tz is not None else idx   # naive index
            return c.resample('2h').last().dropna()

        nifty = h2('^NSEI')
        if nifty is not None and len(nifty) > 200:
            vix = h2('^INDIAVIX')
            vix = (vix.reindex(nifty.index).ffill().bfill()
                   if vix is not None and len(vix) else pd.Series(15.0, index=nifty.index))
            nifty.name, vix.name = 'nifty', 'vix'
            print(f'yfinance 60m->2h: {len(nifty)} bars')
            return nifty, vix, False
        raise ValueError('too few bars')
    except Exception as e:
        print(f'yfinance unavailable ({e}); falling back to synthetic 2h data.')
        return _synth()


def _synth():
    """Regime-blocked GBM + OU VIX at 2h cadence (offline validation only)."""
    rng = np.random.default_rng(42)
    days = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=520)
    idx = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in days for h in (10, 12, 14)])
    REG = [(0.00, 0.15,  0.0004, 0.0035, 14.0, 0.6),
           (0.15, 0.30, -0.0006, 0.0060, 22.0, 1.1),
           (0.30, 0.45,  0.0001, 0.0030, 16.0, 0.7),
           (0.45, 0.55, -0.0030, 0.0110, 45.0, 2.5),
           (0.55, 0.75,  0.0006, 0.0050, 20.0, 1.0),
           (0.75, 0.90,  0.0005, 0.0032, 13.5, 0.6),
           (0.90, 1.00, -0.0002, 0.0045, 18.0, 0.9)]
    n, lvl, pv, nv, vv = len(idx), 18000.0, 15.0, [], []
    for i in range(n):
        f = i / n
        mu, sg, vm, vf = 0.0002, 0.0040, 16.0, 0.8
        for fs, fe, m, s, v, q in REG:
            if fs <= f < fe:
                mu, sg, vm, vf = m, s, v, q
                break
        lvl *= np.exp(rng.normal(mu, sg))
        pv = max(pv + 0.05 * (vm - pv) + rng.normal(0, vm * 0.08 * vf), 8.0)
        nv.append(lvl); vv.append(pv)
    return (pd.Series(nv, index=idx, name='nifty'),
            pd.Series(vv, index=idx, name='vix'), True)


nifty, vix, TAC_SYNTH = load_2h()
if TAC_SYNTH:
    print(f"\n*** TAC_SYNTH = True  ->  SYNTHETIC data ({len(nifty)} bars). "
          f"Results below are ILLUSTRATIVE ONLY. ***\n")
else:
    print(f"\n*** TAC_SYNTH = False  ->  REAL yfinance data ({len(nifty)} bars). "
          f"Results below are decision-grade. ***\n")
print(f"Span: {nifty.index[0]}  ->  {nifty.index[-1]}")

In [ ]:
# ==========================================================================
# FEATURE + LABELING ENGINE -- INLINED VERBATIM (master code cell 7).
# ==========================================================================
def build_features(nifty, vix, scale=LOOKBACK_SCALE):
    """The 9 causal swing features (engine column names)."""
    w = {k: max(2, int(round(v * scale))) for k, v in BASE_WIN.items()}
    r = np.log(nifty / nifty.shift(1))
    df = pd.DataFrame(index=nifty.index)
    df['ret_2h'] = r
    df['mom_1d'] = r.rolling(w['MOM_1D']).sum()
    df['mom_3d'] = r.rolling(w['MOM_3D']).sum()
    df['mom_5d'] = r.rolling(w['MOM_5D']).sum()
    df['vol_2h'] = r.rolling(w['VOL_WIN']).std()
    df['vol_expansion'] = r.rolling(w['VOL_FAST']).std() / r.rolling(w['VOL_SLOW']).std()
    df['vix_chg'] = (vix - vix.shift(1)) / vix.shift(1)
    sh = nifty.rolling(w['SWING_WIN']).max()
    df['drawdown'] = (sh - nifty) / sh
    ma = nifty.rolling(w['SWING_WIN']).mean()
    df['dist_ma'] = (nifty - ma) / ma
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df


def direction_weight(feat, exclude=None):
    """Weight this feature contributes to the composite BULLISHNESS score.

    FIX 1: features listed in `exclude` (default DIRECTION_EXCLUDE) get weight
    0.0 -- they stop voting on DIRECTION while remaining full HMM inputs.
    vol_2h and vol_expansion are magnitude measures, not signed ones; scoring a
    high-volatility RALLY as bearish was the bug this removes.

    Subset-agnostic: excluding a feature that is not in the active subset is a
    no-op, and `exclude=()` reproduces the original weights exactly.
    """
    exclude = DIRECTION_EXCLUDE if exclude is None else exclude
    if feat in exclude:
        return 0.0
    return FEATURE_SIGN[feat] * FEATURE_MAG.get(feat, 1.0)


def composite_subset(means, feature_subset, exclude=None):
    """Bullishness score per state; works with ANY feature subset."""
    score = np.zeros(means.shape[0])
    for j, feat in enumerate(feature_subset):
        score += direction_weight(feat, exclude) * means[:, j]
    # A subset whose every feature is excluded would make all states score 0 and
    # the direction ranking arbitrary -- catch that rather than emit noise.
    assert any(direction_weight(f, exclude) != 0.0 for f in feature_subset), \
        'DIRECTION_EXCLUDE removed every feature from the direction score'
    return score


def n_params(N, F, covariance_type):
    """GaussianHMM free parameters: means + covariances + transitions + startprob."""
    cov = N * F * (F + 1) / 2 if covariance_type == 'full' else N * F
    return N * F + cov + N * (N - 1) + (N - 1)


def trend_efficiency(close, win=EFF_WIN):
    """Causal Kaufman efficiency ratio over `win` bars:

        |close[t] - close[t-win]| / sum(|close.diff()|)

    i.e. net displacement / total path walked. ~1.0 in a straight-line trend,
    ~0.0 when price keeps doubling back inside a range -- the distinction raw
    momentum magnitude cannot make. Uses only bars <= t.
    """
    net = (close - close.shift(win)).abs()
    path = close.diff().abs().rolling(win).sum()
    return (net / path.replace(0, np.nan)).fillna(0.0).clip(0.0, 1.0)


def gate_band(trend_raw, feat, n_fit, mode=None, thr_enter=None, thr_exit=None,
              target_rate=None, exit_slack=None, hysteresis=None):
    """ONE band contract for BOTH intensity modes.

    Returns `(trend, thr_enter, thr_exit)`: the trend-MAGNITUDE series and the
    (enter, exit) thresholds it is graded against.

    THIS FUNCTION EXISTS TO RESOLVE A REAL COUPLING. In the prototype, the gate
    BAND (FIX 3's Z_HI_EXIT / EFF_HI_EXIT) and the INTENSITY MODE (FIX 5) were
    mutually exclusive: the vol_norm branch asserted `z_exit is None`, because
    z_exit was a threshold expressed in frozen-z units and vol_norm derives its
    thresholds by occupancy instead. That is a units problem, not a logic
    problem, and it is fixed here by making the BAND -- not the threshold
    constants -- the shared abstraction. Both modes now:

      * produce a trend series in their OWN units,
      * carry a DEFAULT (enter, exit) band in those same units,
      * accept an OCCUPANCY-derived band (`target_rate` / `exit_slack`) computed
        on the FIT WINDOW of that same series,
      * accept EXPLICIT overrides (`thr_enter` / `thr_exit`) in those same units,
      * accept `hysteresis=False`, a MODE-INDEPENDENT way to say "no band"
        (exit == enter), which is what FIX-3-off means in either mode.

    So the mode and the band are now independent knobs, and nothing is papered
    over with an assert.

    mode='frozen_z' : trend = (mom_3d - mu_fit)/sd_fit.
                      Default band = (Z_HI, Z_HI_EXIT), the pre-change constants.
    mode='vol_norm' : trend = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the
                      t-statistic of the 9-bar move: dimensionless,
                      contemporaneous, needing NO fit-window baseline.
                      Default band = occupancy quantiles at H_TARGET_RATE /
                      H_TARGET_RATE + H_EXIT_SLACK.

    PRECEDENCE (most specific wins): explicit thr_* > occupancy (target_rate /
    exit_slack, honoured under EITHER mode) > the mode default. `hysteresis=False`
    is applied last and collapses exit onto enter whatever produced them.

    CAUSALITY. Under frozen_z, mu/sd come from the leading n_fit bars. Under
    vol_norm the SERIES needs no fit window at all (mom_3d and vol_2h are both
    trailing rolling windows at bar t) and the THRESHOLDS are quantiles over the
    leading n_fit bars only -- computed ONCE, never per bar, never over bars the
    model has not seen. Either way a prefix truncation at any t >= n_fit
    reproduces both the series value at t and the thresholds exactly. Section 7.0
    proves this by truncation rather than asserting it here.
    """
    mode = INTENSITY_MODE if mode is None else mode
    assert mode in ('frozen_z', 'vol_norm'), f'unknown INTENSITY_MODE {mode!r}'
    tr = np.asarray(trend_raw, dtype=float)
    n_fit = int(n_fit)
    assert 2 <= n_fit <= len(tr), 'fit window out of range for the intensity gate'

    if mode == 'frozen_z':
        mu = float(np.mean(tr[:n_fit]))
        sd = float(np.std(tr[:n_fit]))
        trend = (tr - mu) / (sd if sd > 0 else 1.0)
        ok = np.isfinite(trend)
        d_enter, d_exit = float(Z_HI), float(Z_HI_EXIT)
    else:
        assert feat is not None and 'vol_2h' in getattr(feat, 'columns', []), \
            "vol_norm needs the raw feature frame (for vol_2h)"
        vol = np.asarray(feat['vol_2h'].values, dtype=float)
        assert len(vol) == len(tr), 'vol_2h must be aligned 1:1 with the trend feature'
        # DENOMINATOR GUARD: vol_2h is NaN through warm-up and 0.0 on a dead-flat
        # stretch. Those bars get trend = 0.0 -- the neutral value, which fires no
        # gate and can only SUPPRESS an escalation, never invent one.
        den = vol * np.sqrt(MOM_3D_BARS)
        ok = np.isfinite(den) & (den > 0.0) & np.isfinite(tr)
        trend = np.zeros(len(tr), dtype=float)
        np.divide(tr, den, out=trend, where=ok)
        trend[~np.isfinite(trend)] = 0.0
        d_enter = d_exit = None                 # derived by occupancy below

    # ---- occupancy-derived band (either mode) -----------------------------
    if d_enter is None or target_rate is not None or exit_slack is not None:
        tgt = H_TARGET_RATE if target_rate is None else float(target_rate)
        slk = H_EXIT_SLACK if exit_slack is None else float(exit_slack)
        assert 0.0 < tgt < 1.0 and slk >= 0.0, 'target occupancy out of range'
        a = np.abs(trend[:n_fit])[ok[:n_fit]]   # guarded bars excluded from the quantile
        assert a.size >= 20, 'too few usable fit-window bars to derive a threshold'
        d_enter = float(np.quantile(a, 1.0 - tgt))
        d_exit = float(np.quantile(a, 1.0 - min(tgt + slk, 0.999)))

    en = float(d_enter) if thr_enter is None else float(thr_enter)
    ex = float(d_exit) if thr_exit is None else float(thr_exit)
    if hysteresis is False:
        ex = en                                 # FIX-3-OFF, stated mode-independently
    ex = min(ex, en)                            # the exit band must be the looser one
    return trend, en, ex


def intensity_state(z, eff, z_hi=None, eff_hi=None, z_exit=None, eff_exit=None,
                    block_enter=None, force_exit=None):
    """FIX 3 -- gate hysteresis. Returns a signed intensity array:

        +1  bar is escalated H on the BULL side
        -1  bar is escalated H on the BEAR side
         0  bar stays L

    Enter (0 -> +/-1): |z| >= z_hi AND eff >= eff_hi          (unchanged gates)
    Hold  (stay +/-1): |z| >= z_exit AND eff >= eff_exit AND sign(z) unchanged
    Exit  (-> 0):      |z| <  z_exit OR  eff <  eff_exit OR  sign(z) flipped

    A sign flip forces a fresh ENTER test rather than silently relabelling a held
    bull escalation as a bear one.

    CAUSAL: a single left-to-right scan whose state at bar t depends only on
    bars <= t, so truncating the input after t cannot change out[t].

    z_exit == z_hi and eff_exit == eff_hi reduce this EXACTLY to the old
    memoryless `(|z| >= z_hi) & (eff >= eff_hi)` test.

    ESCALATION_DURING_HOLD support -- two OPTIONAL per-bar boolean masks, applied
    INSIDE this same single pass so the state machine stays consistent (a
    suppressed escalation must not be silently "held" on a later bar as though it
    had happened):

      block_enter[t] : the ENTER test is skipped at bar t. An already-running
                       escalation is unaffected and is still held under the exit
                       band. This is 'block'.
      force_exit[t]  : the state is additionally forced to 0 at bar t, ending the
                       run. Re-escalation later must clear the full ENTER band
                       again. This is the extra half of 'demote'.

    Both masks default to all-False, in which case every branch below is exactly
    the pre-change scan -- and Section 7H-viii asserts that bit-for-bit rather
    than trusting this paragraph. Neither mask can CREATE an escalation; both can
    only suppress one, so no chop-filter invariant can be weakened by them.

    CAUSALITY IS PRESERVED BY CONSTRUCTION: mask[t] is consumed at step t of a
    left-to-right scan, so out[t] still depends only on (z, eff, masks)[0..t].
    """
    z_hi = Z_HI if z_hi is None else z_hi
    eff_hi = EFF_HI if eff_hi is None else eff_hi
    z_exit = Z_HI_EXIT if z_exit is None else z_exit
    eff_exit = EFF_HI_EXIT if eff_exit is None else eff_exit
    z = np.asarray(z, dtype=float)
    eff = np.asarray(eff, dtype=float)
    n = len(z)
    _blk = (np.zeros(n, dtype=bool) if block_enter is None
            else np.asarray(block_enter, dtype=bool))
    _fex = (np.zeros(n, dtype=bool) if force_exit is None
            else np.asarray(force_exit, dtype=bool))
    assert len(_blk) == n and len(_fex) == n, 'gate suppression masks must align 1:1 with z'
    out = np.zeros(n, dtype=int)
    state = 0
    for t in range(n):
        s = 1 if z[t] > 0 else (-1 if z[t] < 0 else 0)
        if state != 0 and s == state:
            if abs(z[t]) < z_exit or eff[t] < eff_exit:
                state = 0                       # de-escalate on the EXIT band
        else:
            state = 0                           # no state, or the sign flipped
        if _fex[t]:
            state = 0                           # 'demote': drop a held H to L
        if (state == 0 and not _blk[t]
                and abs(z[t]) >= z_hi and eff[t] >= eff_hi):
            state = s                           # escalate on the ENTER band
        out[t] = state
    return out


def hold_masks(dir_raw, dir_emit, policy=None):
    """(block_enter, force_exit) for ESCALATION_DURING_HOLD.

    A bar is CONTESTED when the emitted direction is not the direction the
    current bar's own evidence votes for -- i.e. `dir_raw[t] != dir_emit[t]`,
    which is exactly the set of bars CONFIRM_BARS is holding through. Both inputs
    are computable from bars <= t, so the mask is causal.

    'allow'  -> (all False, all False)  == unchanged behaviour, bit-for-bit.
    'block'  -> (contested, all False)  == no NEW escalation while contested.
    'demote' -> (contested, contested)  == also drop an existing H to L.
    """
    policy = ESCALATION_DURING_HOLD if policy is None else policy
    assert policy in ('allow', 'block', 'demote'), f'unknown ESCALATION_DURING_HOLD {policy!r}'
    n = len(dir_raw)
    contested = np.asarray(dir_raw) != np.asarray(dir_emit)
    if policy == 'allow':
        z = np.zeros(n, dtype=bool)
        return z, z.copy(), contested
    if policy == 'block':
        return contested, np.zeros(n, dtype=bool), contested
    return contested, contested.copy(), contested


def confirm_delay(raw, confirm_bars=None):
    """FIX 2 -- causal label hysteresis (a CONFIRMATION DELAY, not a smoother).

    A candidate value must be observed on `confirm_bars` CONSECUTIVE bars before
    the emitted series is allowed to flip to it; until then the previous emitted
    value is held.

    WHY THIS IS CAUSAL, stated precisely: out[t] is a function of raw[0..t] only.
    The scan never looks at raw[t+1..]. Contrast with the look-ahead smoother
    this project previously removed, which decided whether to erase a run by
    inspecting that run's full REALIZED length -- i.e. it needed bars after t to
    decide bar t. This does the opposite: it PAYS a delay rather than borrowing
    the future. A flip that turns out to be a 1-bar blip is simply never emitted;
    a flip that persists is emitted `confirm_bars - 1` bars late.

    confirm_bars = 1 reproduces the input exactly (out is raw).
    """
    confirm_bars = CONFIRM_BARS if confirm_bars is None else int(confirm_bars)
    assert confirm_bars >= 1, 'CONFIRM_BARS must be >= 1'
    raw = np.asarray(raw)
    n = len(raw)
    out = np.empty(n, dtype=raw.dtype)
    if n == 0:
        return out
    out[0] = raw[0]                 # bar 0 has no prior emitted label to hold
    cand, run = raw[0], 0
    for t in range(1, n):
        if raw[t] == out[t - 1]:
            out[t] = raw[t]         # agrees with what is already emitted
            cand, run = raw[t], 0
        else:
            if raw[t] == cand:
                run += 1
            else:
                cand, run = raw[t], 1
            if run >= confirm_bars:
                out[t] = cand       # candidate confirmed -> flip
                run = 0
            else:
                out[t] = out[t - 1]  # not yet confirmed -> HOLD the old label
    return out


def direction_buckets(means, feature_subset, exclude=None):
    """Bucket HMM states into bear(-1) / side(0) / bull(+1) by RANK of composite
    bullishness. Rank-based (not a sign+deadzone threshold) so the side bucket is
    guaranteed non-empty; a deadzone can degenerate to an empty side bucket, which
    silently makes SIDEWAYS unreachable. For N=5 this is: bottom 2 bear, middle 1
    side, top 2 bull.

    `exclude` is threaded to the scorer (FIX 1); None uses DIRECTION_EXCLUDE.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    order = np.argsort(scores)                 # ascending bullishness
    n_side = max(1, round(n_st / 5))
    n_bear = (n_st - n_side) // 2
    direction = np.empty(n_st, dtype=int)
    direction[order[:n_bear]] = -1
    direction[order[n_bear:n_bear + n_side]] = 0
    direction[order[n_bear + n_side:]] = 1
    # INVARIANT: every direction bucket must be reachable, else whole labels vanish.
    assert (direction == 1).sum() >= 1, 'bull bucket empty -> H_BULL/L_BULL unreachable'
    assert (direction == -1).sum() >= 1, 'bear bucket empty -> H_BEAR/L_BEAR unreachable'
    assert (direction == 0).sum() >= 1, 'side bucket empty -> SIDEWAYS unreachable'
    return direction


def bar_direction_score(dir_feats, n_fit, features=None):
    """FIX 4 -- CAUSAL PER-BAR directional z-score.

    dir_feats : DataFrame of RAW (unscaled) features aligned to the labeled bars,
                one row per bar, in bar order. Only the signed directional columns
                are read.
    n_fit     : number of LEADING bars that constitute the fit window. The mu/sd
                baseline is computed from those rows ONLY -- exactly the way
                trend_z's baseline is frozen -- so no bar > t and no bar the model
                has not seen can influence bar t's score.

    Construction:
      1. keep only BAR_DIR_FEATURES that are present (ret_2h, mom_1d, mom_3d,
         mom_5d, dist_ma -- all SIGNED directional measures);
      2. z-score each against its FIT-WINDOW mu/sd;
      3. weighted mean with the existing FEATURE_SIGN * FEATURE_MAG weights,
         normalised by sum|w| so the composite stays on a z-like scale;
      4. re-standardize the composite against ITS fit-window mu/sd, so the output
         is a unit-variance z on the fit window and BAR_DIR_TAU is interpretable.

    vol_2h / vol_expansion are structurally barred: they measure how BIG a move
    is, not which way it points, and letting a magnitude term vote on direction
    is the category error this whole fix exists to undo. Asserted below.

    Causality: every input column is a backward-looking rolling statistic, and
    the baseline uses leading rows only, so score[t] depends on bars <= t alone.
    Recomputing from the prefix dir_feats.iloc[:t+1] reproduces score[t] exactly
    (probed in 7.0).
    """
    features = BAR_DIR_FEATURES if features is None else tuple(features)
    assert not (set(features) & {'vol_2h', 'vol_expansion'}), \
        'per-bar DIRECTION score must not contain magnitude features'
    cols = [f for f in features if f in dir_feats.columns]
    assert cols, 'no directional features available for the per-bar direction score'
    A = np.asarray(dir_feats[cols].values, dtype=float)
    assert n_fit >= 2 and n_fit <= len(A), 'fit window out of range for the per-bar score'
    mu = A[:n_fit].mean(axis=0)
    sd = A[:n_fit].std(axis=0)
    Z = (A - mu) / np.where(sd > 0, sd, 1.0)
    # exclude=() deliberately: DIRECTION_EXCLUDE is FIX 1's state-level knob and
    # must not be able to mute a feature that is already guaranteed directional.
    w = np.array([direction_weight(f, exclude=()) for f in cols], dtype=float)
    assert np.abs(w).sum() > 0, 'per-bar direction weights are all zero'
    # (Z * w).sum(axis=1), NOT Z @ w. The two are algebraically identical and the
    # matmul is the obvious way to write it -- but `DataFrame.values` hands back an
    # F-ORDERED array, and BLAS gemv on an F-ordered operand picks its blocking
    # from the ROW COUNT, so the last-bar result changes in the last ulp depending
    # on how many bars follow it. That is a ~1e-16 difference with no economic
    # meaning, but it makes the truncation probe in 7.0 fail its bit-exactness
    # test, and a causality probe that has to be run at a tolerance is a weaker
    # probe. The row-wise form sums 5 terms per row independently of the array
    # length, so score[t] is BIT-identical whether or not bars > t exist -- and
    # 7.0 can therefore assert exact equality rather than np.isclose.
    s = (Z * w).sum(axis=1) / np.abs(w).sum()
    s_mu = float(s[:n_fit].mean())
    s_sd = float(s[:n_fit].std())
    return (s - s_mu) / (s_sd if s_sd > 0 else 1.0)


def bar_direction_masses(score, tau=None):
    """Map the per-bar directional z to bull / side / bear masses that sum to 1.

    Softmax over the three logits (+s/tau, 0, -s/tau): bull dominates for s >> 0,
    bear for s << 0, and SIDE is the plurality only near s ~ 0 -- which is the
    right shape, because "no clear direction at this bar" is a real answer and
    must remain reachable. Computed in a shift-stabilised form so large |s| does
    not overflow.

    Returns (bull, side, bear), each an array over bars, summing to 1 per bar.
    """
    tau = BAR_DIR_TAU if tau is None else float(tau)
    e = np.asarray(score, dtype=float) / max(tau, 1e-12)
    a = np.abs(e)                                   # = max(e, 0, -e), the shift
    eb, es, er = np.exp(e - a), np.exp(-a), np.exp(-e - a)
    tot = eb + es + er
    bull, side, bear = eb / tot, es / tot, er / tot
    assert np.allclose(bull + side + bear, 1.0, atol=1e-9), \
        'per-bar direction masses must partition to 1'
    return bull, side, bear


def _filtered_posteriors(model, X):
    """
    CAUSAL (filtered) state posteriors: P(state_t | observations_1..t).

    Why this exists instead of model.predict_proba():
      hmmlearn's predict_proba runs forward-BACKWARD, so the posterior it
      reports for bar t is smoothed using the whole sequence — including bars
      AFTER t. That is legitimate for offline sequence analysis but is
      look-ahead for a trading regime label: on 2h Nifty data it changes the
      winning state on ~6% of bars versus what was actually knowable at the
      time. predict() (Viterbi) has the same whole-sequence property.

      This is the forward (alpha) recursion only, so each bar's posterior is
      conditioned solely on information available at that bar — exactly what a
      live engine would have. The last bar of a forward-backward pass happens
      to equal the filtered value (no future exists yet), which is why LIVE
      calls were always correct; it is the HISTORICAL labels, and therefore
      every backtest built on them, that needed this fix.

    Computed by the SCALED forward algorithm: alpha is held in LINEAR space and
    renormalised to sum 1 at every step. See `_filtered_posteriors_logspace`
    below for the original log-space/logsumexp formulation, which this is checked
    against bar-by-bar in Section 7.0.

    WHY THE SCALED FORM IS THE SAME ANSWER. The quantity wanted here is the
    NORMALISED filtered posterior at each t, which is scale-free in alpha: for
    any c_t > 0, normalising c_t * alpha_t gives the identical row. So the
    per-step renormalisation -- which is what the log-space version was already
    doing, just via logsumexp -- is not an approximation, it IS the answer. The
    emission frame is likewise exponentiated after subtracting its per-row max,
    another positive per-row constant that cancels in the same normalisation.
    Nothing accumulates, so nothing underflows: alpha sums to 1 after every bar.

    WHY IT IS FASTER. The recursion over t is inherently sequential and is NOT
    vectorised across t (doing so would be wrong). What changes is the cost of
    each step: two scipy.special.logsumexp calls plus an (N,N) broadcast add and
    an exp become one length-N matrix-vector product, one multiply and one
    divide. logsumexp is a Python-level function doing max/subtract/exp/sum/log
    over an (N,N) array per bar; on ~2.9k bars per model per labeling call, and
    hundreds of such calls, that dominates the labeling cost.

    A guard falls back to the log-space implementation if the linear recursion
    ever produces a non-finite or non-positive normaliser, so the fast path can
    never silently return a degraded answer.
    """
    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    start     = model.startprob_ + tiny
    trans     = model.transmat_ + tiny

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    # exp of the emission log-likelihoods, per-bar max removed. The removed
    # factor is a positive per-row constant and cancels in the normalisation.
    frame = np.exp(log_frame - log_frame.max(axis=1, keepdims=True))

    alpha = start * frame[0]
    s = alpha.sum()
    if not (s > 0.0 and np.isfinite(s)):
        return _filtered_posteriors_logspace(model, X)
    alpha = alpha / s
    out[0] = alpha

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        alpha = (alpha @ trans) * frame[t]
        s = alpha.sum()
        if not (s > 0.0 and np.isfinite(s)):
            return _filtered_posteriors_logspace(model, X)
        alpha = alpha / s                                 # renormalise each step
        out[t] = alpha

    return out


def _filtered_posteriors_logspace(model, X):
    """The ORIGINAL log-space / logsumexp forward recursion.

    Kept verbatim as (a) the reference implementation that
    `_filtered_posteriors` is asserted equal to in Section 7.0, and (b) the
    fallback if the scaled recursion ever hits a degenerate normaliser. Slower,
    but identical in what it computes.
    """
    from scipy.special import logsumexp

    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    log_start = np.log(model.startprob_ + tiny)
    log_trans = np.log(model.transmat_ + tiny)

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    log_alpha = log_start + log_frame[0]
    log_alpha -= logsumexp(log_alpha)
    out[0] = np.exp(log_alpha)

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        log_alpha = logsumexp(log_alpha[:, None] + log_trans, axis=0) + log_frame[t]
        log_alpha -= logsumexp(log_alpha)                 # renormalise each step
        out[t] = np.exp(log_alpha)

    return out


def ensemble_seeds(K=None, base_seed=None):
    """Deterministic seed list for the ensemble: base_seed + 0..K-1."""
    K = ENSEMBLE_K if K is None else int(K)
    base_seed = BASE_SEED if base_seed is None else int(base_seed)
    return [base_seed + i for i in range(K)]


def _fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd):
    """ONE ensemble member. Top-level (not a closure) so joblib can pickle it.

    Fully determined by its arguments: the seed is explicit, so this touches no
    global RNG state and is identical whether it runs in this process or a
    worker. This is the only place a GaussianHMM is constructed and fit.
    """
    m = hmm.GaussianHMM(n_components=n_components, covariance_type=covariance_type,
                        n_iter=n_iter, random_state=sd,
                        init_params='stmc', params='stmc')
    m.fit(X_train)
    return m


def fit_hmm_ensemble(X_train, n_components, covariance_type,
                     K=None, base_seed=None, n_iter=None):
    """Fit K GaussianHMMs on the SAME training slice with K different seeds.

    This is the identifiability fix. EM is a local optimizer; one seed gives one
    arbitrary local optimum. K seeds give K samples of the optimum set, whose
    direction-bucket masses are averaged in `ensemble_direction_masses` below.

    Every model sees EXACTLY the same rows (`X_train`), which must already be
    the causal leading slice -- this function does no slicing of its own, so it
    cannot introduce look-ahead.

    The K fits are INDEPENDENT and each is fully determined by its own
    random_state, so with N_JOBS != 1 they are dispatched concurrently via
    joblib. That is a pure scheduling change: no fit can observe another, and
    none of them consumes global RNG state (each gets an explicit seed). N_JOBS=1
    takes the plain serial loop. Section 7.0 asserts the two paths return
    BIT-IDENTICAL models.

    Returns (models, all_converged).
    """
    n_iter = HMM_ITER if n_iter is None else int(n_iter)
    seeds = ensemble_seeds(K, base_seed)

    if N_JOBS == 1 or len(seeds) == 1:
        models = [_fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd)
                  for sd in seeds]
    else:
        from joblib import Parallel, delayed
        models = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(_fit_one_hmm)(X_train, n_components, covariance_type, n_iter, sd)
            for sd in seeds)

    all_conv = all(bool(m.monitor_.converged) for m in models)
    return list(models), all_conv


def ensemble_direction_masses(models, Xs, feature_subset, exclude=None):
    """Average the direction-bucket probability masses across an ensemble.

    For each fitted model:
      1. CAUSAL filtered posteriors via `_filtered_posteriors` (forward-only
         alpha recursion). Never predict/predict_proba over the whole sequence --
         those are forward-BACKWARD/Viterbi and smooth bar t with bars after t.
      2. `direction_buckets` maps that model's states to bear/side/bull.
      3. The per-state posterior collapses to 3 columns: bull / side / bear.

    Those 3-column arrays are then averaged across models. This is only valid
    because direction masses are PERMUTATION-INVARIANT: model A's "state 3" and
    model B's "state 1" are unrelated integers, but "the probability mass sitting
    in bullish states" means the same thing in both. Averaging raw per-state
    posteriors would be meaningless.

    Causality is preserved exactly: the average of K quantities each of which
    depends only on bars <= t depends only on bars <= t.

    Returns (bull_mass, side_mass, bear_mass, probs_model0), where probs_model0
    is the first model's per-state filtered posterior (used only for the
    `hmm_state_int` reporting column, which has no ensemble analogue).
    """
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        direction = direction_buckets(m.means_, feature_subset, exclude)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc[:, 0] += probs[:, direction == 1].sum(axis=1)
        acc[:, 1] += probs[:, direction == 0].sum(axis=1)
        acc[:, 2] += probs[:, direction == -1].sum(axis=1)
    acc /= len(models)
    # INVARIANT: an average of rows that each sum to 1 must itself sum to 1.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE = 'soft'. IMPLEMENTED AND SWITCHABLE, NOT ADOPTED.
#
# READ THIS BEFORE TURNING IT ON. Soft bucketing improves stability AT THE SOURCE
# (the state -> direction map stops flipping wholesale when one state's composite
# score crosses another's -- measured 3.2x more stable) but it makes the EMITTED
# SIDEWAYS / BEAR occupancy gaps WORSE. The mechanism is the standardization
# below: with only N = 5 states the composite scores are standardized by the sd
# of those same 5 numbers, so a single outlier state inflates the sd and drags
# every other state's z toward zero, washing the map toward SIDEWAYS by a
# different amount in each fit. IT IS NOT RECOMMENDED UNTIL THE STANDARDIZATION
# IS FIXED (DIR_SCALE='mad' is the obvious first thing to try). DIRECTION_MODE
# stays 'rank'.
# ---------------------------------------------------------------------------
def _sigmoid(x):
    """Overflow-free logistic. exp is evaluated only on the non-positive side."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    p, n = x >= 0, x < 0
    out[p] = 1.0 / (1.0 + np.exp(-x[p]))
    e = np.exp(x[n])
    out[n] = e / (1.0 + e)
    return out


def soft_direction_weights(means, feature_subset, exclude=None, tau=None, c=None,
                           require_reachable=True, scale=None):
    """SOFT replacement for `direction_buckets`. Returns (W, z, scores).

    W is an (N, 3) row-stochastic matrix with columns [bull, side, bear]. The
    scorer is `composite_subset` -- the notebook's own, unchanged -- so FIX 1
    (`DIRECTION_EXCLUDE`) is threaded through untouched and 'soft' reads exactly
    the same evidence 'rank' does. Only the mapping score -> bucket changes: a
    step function of the RANK becomes a smooth function of the VALUE.

    Standardizing ACROSS STATES (not across bars) is what makes DIR_TAU / DIR_C
    scale-free -- and is also the weakness described in the block comment above.

    require_reachable : enforce that no bucket is structurally dead (the invariant
    the reverted sign+deadzone attempt violated). For c > 0 and tau > 0 this holds
    on every state in exact arithmetic; it can only fail when the sigmoids
    SATURATE in floating point, i.e. as tau -> 0, where the rule degenerates back
    into that deadzone.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    assert n_st >= 3, 'need at least 3 states for a 3-bucket direction map'
    _scl = DIR_SCALE if scale is None else scale
    if _scl == 'sd':
        ctr, sd = float(np.mean(scores)), float(np.std(scores))
    else:
        assert _scl == 'mad', f'unknown DIR_SCALE {_scl!r}'
        ctr = float(np.median(scores))
        sd = 1.4826 * float(np.median(np.abs(scores - ctr)))
        if sd <= 0:                       # >= half the states tied: fall back
            ctr, sd = float(np.mean(scores)), float(np.std(scores))
    z = (scores - ctr) / (sd if sd > 0 else 1.0)

    tau = DIR_TAU if tau is None else float(tau)
    c = DIR_C if c is None else float(c)
    t = max(tau, 1e-12)                      # tau -> 0 becomes a hard threshold

    w_bull = _sigmoid((z - c) / t)
    w_bear = _sigmoid((-z - c) / t)
    w_side = np.maximum(0.0, 1.0 - w_bull - w_bear)

    W = np.stack([w_bull, w_side, w_bear], axis=1)
    tot = W.sum(axis=1)
    assert (tot > 0).all(), 'a state ended up with zero weight in all three buckets'
    W = W / tot[:, None]
    # EXACT partition: after the division the row sum is 1 only to within a ulp,
    # so the residual is handed to the row's LARGEST component (>= 1/3, so
    # `1 - rest` stays safely positive and non-negativity survives).
    for i in range(n_st):
        j = int(np.argmax(W[i]))
        others = [k for k in range(3) if k != j]
        W[i, j] = 1.0 - (W[i, others[0]] + W[i, others[1]])
    assert (np.abs(W.sum(axis=1) - 1.0) <= 4 * np.finfo(float).eps).all(), \
        'per-state weights must partition to 1'
    assert (W >= 0.0).all(), 'per-state weights must be non-negative'
    if require_reachable:
        assert W[:, 1].max() > 0.0, \
            'SIDE weight is zero on every state -> SIDEWAYS unreachable (the deadzone bug)'
        assert W[:, 0].max() > 0.0, 'BULL weight is zero on every state'
        assert W[:, 2].max() > 0.0, 'BEAR weight is zero on every state'
    return W, z, scores


def hard_direction_weights(means, feature_subset, exclude=None):
    """`direction_buckets` expressed as the same one-hot (N, 3) object
    `soft_direction_weights` returns, so the two modes are one construction with
    two settings rather than two code paths. Diagnostics only -- the 'rank'
    labeling path calls `ensemble_direction_masses` itself, so the bit-for-bit
    claim is never routed through this helper."""
    d = direction_buckets(means, feature_subset, exclude)
    W = np.zeros((len(d), 3), dtype=float)
    W[d == 1, 0] = 1.0
    W[d == 0, 1] = 1.0
    W[d == -1, 2] = 1.0
    assert (W.sum(axis=1) == 1.0).all()
    return W, d


def ensemble_direction_masses_by_mode(models, Xs, feature_subset, exclude=None,
                                      direction_mode=None, tau=None, c=None, scale=None):
    """`ensemble_direction_masses` with the state -> bucket map made switchable.

    mode='rank' : DELEGATES to the unchanged function above, so it is bit-for-bit
                  today's behaviour by construction rather than by
                  re-implementation.
    mode='soft' : identical pipeline -- CAUSAL filtered posteriors, per-model
                  collapse to 3 direction columns, average across the ensemble --
                  except the collapse is `probs @ W` instead of summing the
                  columns of a hard partition.

    Averaging across the ensemble stays valid for the same reason it always did:
    direction masses are PERMUTATION-INVARIANT. Causality is untouched: W depends
    on the FITTED MEANS only (fit window) and `probs` is the forward-only
    filtered posterior.
    """
    mode = DIRECTION_MODE if direction_mode is None else direction_mode
    if mode == 'rank':
        assert tau is None and c is None and scale is None, \
            'dir_tau / dir_c / dir_scale are soft-mode knobs and do nothing under rank'
        return ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert mode == 'soft', f'unknown DIRECTION_MODE {mode!r}'
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        W, _z, _sc = soft_direction_weights(m.means_, feature_subset, exclude, tau, c,
                                            scale=scale)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc += probs @ W
    acc /= len(models)
    # INVARIANT: a convex combination of simplex rows is a simplex row.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'soft ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


DIR_REACH_MIN = 0.10   # a direction bucket must carry at least this much mass on
                       # SOME bar to count as reachable. Deliberately a low bar:
                       # the point is to catch a STRUCTURALLY dead bucket (the
                       # deadzone bug), not to legislate an occupancy.


def label_bars(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
               exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
               dir_feats=None, bar_dir_weight=None,
               intensity_mode=None, feat_raw=None, z_enter=None,
               target_rate=None, exit_slack=None, gate_hysteresis=None,
               direction_mode=None, dir_tau=None, dir_c=None, dir_scale=None,
               escalation_during_hold=None):
    """Direction + intensity + chop-filter labeling: an inline port of the
    engine's _fit_and_classify labeling block (no repo import).

    model         : fitted GaussianHMM, OR a list/tuple of them (a seed ensemble).
                    With a list, the bull/side/bear masses are the ENSEMBLE
                    AVERAGE; a single model (or a 1-element list) reproduces the
                    old single-fit behaviour bit for bit. Nothing else about the
                    labeling changes -- the gates, the CONF_L override, the
                    output columns and their semantics are identical either way.
    Xs            : scaled features for bars 0..len(dates)-1 (scaler fit on <= n_fit)
    dates         : DatetimeIndex for those bars
    price         : full close Series (reindexed internally; efficiency is causal)
    trend_raw     : TREND_FEATURE values aligned to `dates`
    n_fit         : number of LEADING bars the model/scaler were fit on -- the trend
                    mu/sd baseline is frozen on exactly this window (no look-ahead)

    The three switchable labeling fixes (all default to the module-level config;
    the values in brackets reproduce the PRE-FIX behaviour bit for bit):

    exclude       : FIX 1, features that do not vote on direction  [()]
    confirm_bars  : FIX 2, causal confirmation delay in bars       [1]
    z_exit,
    eff_exit      : FIX 3, gate de-escalation band                 [Z_HI, EFF_HI]
    dir_feats     : FIX 4, RAW feature frame aligned to `dates` (the per-bar
                    direction score reads BAR_DIR_FEATURES out of it)
    bar_dir_weight: FIX 4, blend weight on the per-bar masses      [0.0]

    And the three switches added in this consolidation (again, the bracketed
    value reproduces the PRE-CHANGE behaviour bit for bit -- asserted against a
    frozen verbatim copy of the old function in Section 7H-viii):

    intensity_mode: FIX 5, 'frozen_z' | 'vol_norm'                 ['frozen_z']
    feat_raw      : FIX 5, raw feature frame (vol_norm reads vol_2h out of it);
                    falls back to `dir_feats`, which is the same frame at every
                    call site that supplies one
    z_enter,
    z_exit        : FIX 5/3, EXPLICIT band overrides, expressed in the units of
                    whichever intensity_mode is in force. These are no longer
                    frozen_z-only knobs -- see `gate_band`
    target_rate,
    exit_slack    : FIX 5/3, band derived by FIT-WINDOW target occupancy; valid
                    under EITHER mode
    gate_hysteresis: FIX 3 as a mode-INDEPENDENT boolean. False collapses exit
                    onto enter in either mode, which is what FIX-3-off means
                                                                   [False]
    direction_mode: FIX 6, 'rank' | 'soft'                         ['rank']
    dir_tau,
    dir_c,
    dir_scale     : FIX 6 soft-mode knobs, rejected under 'rank'
    escalation_during_hold : 'allow' | 'block' | 'demote'          ['allow']

    Returns a DataFrame indexed by `dates` with the engine's column names.
    """
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    # CAUSAL decoding: filtered (forward-only) posteriors, NOT hmmlearn's
    # forward-backward predict_proba / Viterbi predict -- both of those smooth
    # bar t with bars after t, which is look-ahead in a trading label.
    #
    # Aggregate probability mass BY DIRECTION BUCKET, not by individual state: a
    # bull move split across two bullish states must not be diluted below CONF_L
    # and mislabelled SIDEWAYS. With an ensemble, those bucket masses are then
    # AVERAGED over the K fits (permutation-invariant, so this is well defined).
    # FIX 6 rides here: 'rank' delegates to the unchanged ensemble function, so
    # the default path is bit-for-bit what it was.
    _dmode = DIRECTION_MODE if direction_mode is None else direction_mode
    bull_mass, side_mass, bear_mass, probs = ensemble_direction_masses_by_mode(
        models, Xs, feature_subset, exclude, _dmode, dir_tau, dir_c, dir_scale)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'direction masses must partition to 1'

    # ---- FIX 4: blend in the CAUSAL PER-BAR direction ----------------------
    # The masses above are per-STATE evidence: bar t inherits the direction of
    # whichever states it sits in, and a directionally MIXED state (the classic
    # high-volatility state, which holds both sharp selloffs and sharp rallies)
    # hands the same answer to bars pointing opposite ways. The per-bar score is
    # computed from signed features only and asks the question one bar at a time.
    #
    # A convex combination of two 3-simplex points is a 3-simplex point, so the
    # partition-to-1 invariant survives untouched, and so does everything built
    # on it (prob_*, confidence, the CONF_L override, the gates, the hysteresis).
    #
    # w = 0.0 is EXACT: 1.0*m + 0.0*b == m in IEEE754 for finite non-negative m.
    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0, 'BAR_DIR_WEIGHT must be in [0, 1]'
    if dir_feats is None:
        assert _bw == 0.0, \
            'bar_dir_weight > 0 requires dir_feats (the raw feature frame for these bars)'
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates), 'dir_feats must be aligned 1:1 with dates'
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'BLENDED direction masses must still partition to 1'

    # `states` is the FIRST ensemble member's filtered argmax. Raw state indices
    # have no ensemble-wide meaning (they permute between fits), so this column
    # is reporting-only and is never used to form a label. With K=1 it is exactly
    # the old hmm_state_int.
    states = probs.argmax(axis=1)

    # REACHABILITY -- no direction bucket may be structurally dead. This is the
    # invariant the reverted sign+deadzone attempt violated (empty side bucket ->
    # SIDEWAYS unreachable). Checked under BOTH direction modes, on the BLENDED
    # masses, i.e. on what the labels are actually formed from.
    for _nm, _m in (('BULL', bull_mass), ('SIDE', side_mass), ('BEAR', bear_mass)):
        assert float(np.max(_m)) >= DIR_REACH_MIN, \
            f'{_nm} bucket never reaches {DIR_REACH_MIN} mass on any bar -> unreachable'

    n = len(dates)
    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values

    # ---- DIRECTION IS DECIDED FIRST -------------------------------------
    # The intensity gate is computed AFTER the direction now, because
    # ESCALATION_DURING_HOLD needs to know whether the emitted direction is
    # contested before it can decide whether an escalation is allowed. Nothing
    # about the pre-change computation depended on the old order: dir_raw and
    # dir_emit never read the gate, and the gate never read the direction. With
    # ESCALATION_DURING_HOLD='allow' the reorder is a pure no-op, and Section
    # 7H-viii asserts that bit-for-bit against the frozen old function.
    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)              # 0=bull, 1=bear, 2=side (argmax, not "nonzero")

    # DIRECTION first (bull / bear / side), including the CONF_L override, ...
    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)

    # ... then FIX 2, the causal confirmation delay, applied to the DIRECTION.
    #
    # Why direction and not the full 5-label string: an H<->L intensity flicker
    # inside one direction would otherwise keep resetting the direction candidate
    # and can freeze the emitted label indefinitely (raw H_BULL, L_BULL, H_BULL,
    # L_BULL, ... never confirms anything at CONFIRM_BARS=2, so a clean rally
    # would stay stuck on whatever preceded it). Direction flicker is also
    # precisely the barcode the user objected to; H<->L flicker is fix 3's job.
    # confirm_bars=1 leaves dir_emit == dir_raw, i.e. the old behaviour exactly.
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    # ---- THE INTENSITY GATE ---------------------------------------------
    # FIX 5: the trend-magnitude series AND the band it is graded against now
    # come from one place (`gate_band`), so INTENSITY_MODE and the enter/exit
    # band are independent knobs rather than mutually exclusive ones.
    #
    # FIX 3 lives inside that band: escalation requires |z| >= thr_enter AND
    # eff >= EFF_HI; de-escalation requires falling below the LOOSER exit band,
    # so a bar hovering at the threshold no longer flickers H/L every bar.
    # gate_hysteresis=False collapses exit onto enter in either mode, which is
    # the pre-FIX-3 memoryless test.
    _imode = INTENSITY_MODE if intensity_mode is None else intensity_mode
    _fr = feat_raw if feat_raw is not None else dir_feats
    z, thr_enter, thr_exit = gate_band(trend_raw, _fr, n_fit, _imode,
                                       z_enter, z_exit, target_rate, exit_slack,
                                       gate_hysteresis)
    assert thr_exit <= thr_enter, 'the exit band must not be tighter than the enter band'

    # ESCALATION_DURING_HOLD: a bar is CONTESTED when this bar's own evidence
    # (dir_raw) disagrees with the direction CONFIRM_BARS is holding (dir_emit).
    # Under 'allow' both masks are all-False and this is a no-op.
    _hold = ESCALATION_DURING_HOLD if escalation_during_hold is None else escalation_during_hold
    _blk, _fex, _contested = hold_masks(dir_raw, dir_emit, _hold)

    intens = intensity_state(z, eff, thr_enter, EFF_HI, thr_exit, eff_exit,
                             block_enter=_blk, force_exit=_fex)
    hi_bull = intens == 1
    hi_bear = intens == -1

    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    # INVARIANT: the 5 prob buckets always partition the full probability mass,
    # independently of the confidence override above.
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6), \
        'prob_* columns must sum to 1'

    # Intensity is then graded at bar t from the (hysteretic) gate state, so an
    # emitted H bar always clears the gates AT THAT BAR -- the chop-filter
    # invariant below is a statement about the label that is actually emitted.
    # np.full/boolean assignment rather than np.select: np.select would type the
    # result from the choicelist (<U6) and silently TRUNCATE 'SIDEWAYS' to
    # 'SIDEWA'. dtype is pinned explicitly here.
    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)     # what the notebook uses everywhere
    raw_state    = _compose(dir_raw)      # pre-confirmation, diagnostics only

    # INVARIANT (chop filter), generalised for the enter/exit bands: no bar may be
    # graded H without clearing the gate that is ACTIVE for it -- the ENTER gate on
    # the first bar of an H run, the (looser) EXIT gate on a bar the run is being
    # held through. Stated against the thresholds ACTUALLY IN FORCE (thr_enter /
    # thr_exit), which is what makes it mode-independent; with hysteresis off the
    # two collapse into the single original assert.
    _ex = EFF_HI_EXIT if eff_exit is None else eff_exit
    is_h = np.isin(regime_state, ['H_BULL', 'H_BEAR'])
    if is_h.any():
        assert (eff[is_h] >= min(EFF_HI, _ex)).all(), 'H bar below the EFF exit band -> chop filter bypassed'
        assert (np.abs(z[is_h]) >= thr_exit).all(), 'H bar below the intensity exit band -> magnitude gate bypassed'
        # and every ESCALATION -- the bar on which the gate state machine turned ON,
        # which is where the ENTER band must have been cleared. (The bar an emitted
        # H *label* run starts on is NOT the right anchor: the direction can flip to
        # BULL several bars into an already-escalated stretch, and that bar only
        # owes the exit band.)
        _prev_i = np.concatenate(([0], intens[:-1]))
        _on = np.flatnonzero((intens != 0) & (intens != _prev_i))   # incl. +1 -> -1 flips
        assert (eff[_on] >= EFF_HI).all(), 'H escalation below EFF_HI -> enter gate bypassed'
        assert (np.abs(z[_on]) >= thr_enter).all(), \
            'H escalation below the intensity enter threshold -> enter gate bypassed'
        # ESCALATION_DURING_HOLD: under 'block'/'demote' no escalation may BEGIN on
        # a contested bar, and under 'demote' no H may be emitted on one at all.
        if _hold in ('block', 'demote'):
            assert not _contested[_on].any(), \
                'a NEW escalation fired on a contested bar -> ESCALATION_DURING_HOLD bypassed'
        if _hold == 'demote':
            assert not (is_h & _contested).any(), \
                'an H label survived on a contested bar under ESCALATION_DURING_HOLD=demote'
    assert set(np.unique(regime_state)).issubset(set(REGIME_LABELS)), 'unknown label emitted'

    out = pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        # diagnostics for Section 7H (never inputs to anything):
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
        # ESCALATION_DURING_HOLD diagnostics (never inputs to anything).
        # `contested` keeps the prototype's column name so the two artifacts
        # can be diffed directly.
        'dir_raw':                    dir_raw,
        'dir_emit':                   dir_emit,
        'contested':                  _contested,
    }, index=dates)
    out.index.name = 'date'
    out.attrs['intensity_mode'] = _imode
    out.attrs['direction_mode'] = _dmode
    out.attrs['escalation_during_hold'] = _hold
    out.attrs['thr_enter'] = float(thr_enter)
    out.attrs['thr_exit'] = float(thr_exit)
    return out


# ---------------------------------------------------------------------------
# THE FROZEN PRE-CHANGE REFERENCE.
#
# `label_bars_legacy` is a VERBATIM copy of `label_bars` as it stood BEFORE this
# consolidation -- before INTENSITY_MODE, DIRECTION_MODE, ESCALATION_DURING_HOLD
# and the direction-before-intensity reorder. It exists for exactly one purpose:
# Section 7H-viii runs both functions over a matrix of argument combinations and
# asserts BIT-FOR-BIT equality of every emitted column whenever the new switches
# sit at their OFF values. That turns "these switches are no-ops when off" from a
# claim in a comment into a test.
#
# It is never called by the pipeline. Do not "improve" it -- its whole value is
# that it is frozen.
# ---------------------------------------------------------------------------
def label_bars_legacy(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
                      exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
                      dir_feats=None, bar_dir_weight=None):
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    bull_mass, side_mass, bear_mass, probs = \
        ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0
    if dir_feats is None:
        assert _bw == 0.0
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates)
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    states = probs.argmax(axis=1)

    trend_raw = np.asarray(trend_raw, dtype=float)
    trend_mu = float(np.mean(trend_raw[:n_fit]))
    trend_sd = float(np.std(trend_raw[:n_fit]))
    z = (trend_raw - trend_mu) / (trend_sd if trend_sd > 0 else 1.0)

    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values
    intens = intensity_state(z, eff, Z_HI, EFF_HI, z_exit, eff_exit)
    hi_bull = intens == 1
    hi_bear = intens == -1

    n = len(dates)
    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6)

    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)

    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)
    raw_state    = _compose(dir_raw)

    return pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
    }, index=dates)


print('features + direction/intensity labeling core ready')
print(f'  gate      : INTENSITY_MODE={INTENSITY_MODE!r}  band via gate_band() '
      f'(mode and band are independent knobs)')
print(f'  direction : DIRECTION_MODE={DIRECTION_MODE!r}   hold policy: '
      f'ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}')
print('  label_bars_legacy (frozen pre-change copy) available for the 7H-viii equivalence test')

In [ ]:
# ==========================================================================
# PLOT HELPERS -- INLINED VERBATIM (master code cell 9).
# ==========================================================================
def regime_blocks(series):
    """[(label, start_ts, end_ts), ...] contiguous runs of the same label."""
    vals, idx = series.values, series.index
    blocks, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            blocks.append((vals[start], idx[start], idx[i - 1]))
            start = i
    blocks.append((vals[start], idx[start], idx[-1]))
    return blocks


def shade_bands(ax, spans, alpha=0.35, zorder=1):
    """Paint (label, x0, x1) spans as ONE PolyCollection PER LABEL.

    Replaces a per-block `ax.axvspan` loop. On this data a chart has 500+
    contiguous regime blocks, so the loop built 500+ individual Patch artists per
    axes and ~13 figures paid for it; this builds at most len(REGIME_LABELS)
    collections instead, with the same geometry.

    Visually identical, by construction rather than by eye:
      * the x-ranges come from the SAME `regime_blocks` output, unchanged;
      * `broken_barh` with `ax.get_xaxis_transform()` is the same blended
        transform `axvspan` uses -- x in DATA coordinates, y in AXES fraction
        0..1 -- so bands span the full height and ignore the y data limits
        exactly as axvspan did;
      * colour, alpha, linewidth=0 and zorder=1 are the axvspan values.

    Grouping by label is safe because `regime_blocks` returns DISJOINT spans, so
    no two bands overlap and the draw order between them cannot matter.

    ------------------------------------------------------------------------
    THE Y-AXIS BUG THIS FIXES (user-visible; NOT reproducible on every
    matplotlib, so it is fixed structurally rather than by chasing a repro).
    ------------------------------------------------------------------------
    On the user's Kaggle matplotlib the shaded price panels came out with the
    y-axis dragged down to 0, squashing the price line into the top fifth of the
    panel. The cause is the blended transform: the band geometry is y = 0..1 in
    AXES-FRACTION coordinates, but `broken_barh` -> `add_collection` defaults to
    `autolim=True`, and an older matplotlib folds the collection's raw y-extent
    (those literal 0 and 1) into the axes DATA limits before the transform is
    considered. The autoscaler then has to fit both `[0, 1]` and `[24000, 26000]`
    and produces `[0, 26000]`.

    Fixed two ways, deliberately belt-and-braces:
      1. HERE -- build the PolyCollection directly and add it with
         `autolim=False`, so it cannot contribute to the datalim on ANY
         matplotlib version. This is the structural fix.
      2. At every call site -- `set_price_ylim` sets the y-limits EXPLICITLY from
         the plotted series, so the autoscaler is never consulted at all.
    Each of the two alone is sufficient; together the panel cannot regress.

    The speed optimization is NOT reverted: this still builds at most
    len(REGIME_LABELS) collections per axes, not one Patch per block.
    """
    import matplotlib.dates as _mdates
    from matplotlib.collections import PolyCollection
    by_lab = {}
    for lb, d0, d1 in spans:
        x0, x1 = _mdates.date2num(d0), _mdates.date2num(d1)
        by_lab.setdefault(lb, []).append((x0, x1))
    for lb, xr in by_lab.items():
        verts = [[(x0, 0.0), (x1, 0.0), (x1, 1.0), (x0, 1.0)] for x0, x1 in xr]
        coll = PolyCollection(verts,
                              facecolors=REGIME_COLORS.get(lb, '#808080'),
                              alpha=alpha, linewidths=0, zorder=zorder)
        # x in DATA coords, y in AXES fraction 0..1 -- the same blended transform
        # axvspan and broken_barh use, so the bands still span the full height.
        coll.set_transform(ax.get_xaxis_transform())
        ax.add_collection(coll, autolim=False)     # <-- cannot touch the datalim


def shade_regimes(ax, series, alpha=0.35):
    shade_bands(ax, regime_blocks(series), alpha=alpha)


# Registry of every shaded panel whose y-limits were set explicitly, so Section
# 7Z can assert -- once, centrally -- that each one brackets its own series and
# excludes 0. A panel that forgot to call this simply never gets checked, so the
# registry is printed with its expected count too.
YLIM_CHECKS = []


def set_price_ylim(ax, series, pad=0.03, tag=''):
    """Set y-limits EXPLICITLY from the plotted series and record the check.

    Never leaves a shaded price/VIX panel to the autoscaler. `pad` is a fraction
    of the series range (falling back to a fraction of the level, then to 1.0,
    for a degenerate flat series).
    """
    v = np.asarray(series, dtype=float)
    v = v[np.isfinite(v)]
    assert v.size, f'set_price_ylim got no finite values ({tag})'
    lo, hi = float(v.min()), float(v.max())
    m = (hi - lo) * pad or abs(hi) * pad or 1.0
    ax.set_ylim(lo - m, hi + m)
    YLIM_CHECKS.append((tag, ax, lo, hi))
    return lo, hi


SPLIT_STYLE = dict(color='blue', linestyle='--', linewidth=1.2, zorder=5)
SPLIT_LABEL = 'train/test split'


def mark_split(ax, index, split_ts):
    """Draw the anchored train/test boundary, if it falls inside this panel.

    Returns the legend handle when the line was drawn and None when the panel's
    window does not contain the split (the zoom panels), so a caller can add the
    legend entry only where there is actually a line to explain.

    THE SPLIT IS A BACKTEST DEVICE. It marks where the fit window ended so that
    bars to its right can be scored on data the model never saw. A LIVE engine
    has no such boundary: it fits on all history to date and classifies the next
    bar. Nothing to the left of this line is "less real" -- it is simply
    in-sample, and therefore not evidence.
    """
    if split_ts is None or len(index) == 0:
        return None
    if not (index[0] <= split_ts <= index[-1]):
        return None
    ax.axvline(split_ts, **SPLIT_STYLE)
    return plt.Line2D([0], [0], color=SPLIT_STYLE['color'],
                      ls=SPLIT_STYLE['linestyle'], label=SPLIT_LABEL)


def regime_legend(ax, loc='upper left', extra=None, **kw):
    handles = [mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r) for r in REGIME_LABELS]
    if extra:
        handles += extra
    ax.legend(handles=handles, loc=loc, fontsize=8, ncol=3, **kw)


def synth_tag():
    return "  [SYNTHETIC DATA - illustrative only]" if TAC_SYNTH else "  [real yfinance data]"

## 2. The four PRE-REGISTERED predictions

Printed **before** any number in this notebook exists, together with the exact
arithmetic that will grade them. Section 8 grades itself against this block and
prints, in plain words, whether the small-sample diagnosis is CONFIRMED or
REFUTED. **If P1 and P2 both fail, the diagnosis is WRONG and the notebook says
so.** No prediction is reinterpreted after its number is seen.

In [ ]:
# ===========================================================================
# PRE-REGISTERED PREDICTIONS + THE GRADING ARITHMETIC. Declared here, first,
# before a single fit is run.
# ===========================================================================
import itertools, time, contextlib
import matplotlib.dates as mdates

T_START = time.time()

P1_S_MIN      = 95.0   # P1: S must exceed this
BIMODAL_SPREAD = 10.0  # a pairwise matrix is BIMODAL if (max-min) >= this pp ...
BIMODAL_GAP    = 5.0   # ... AND the largest gap between sorted pairwise values >= this
BASIN_LOW      = 75.0  # P2: a pairwise value below this is an ACROSS-BASIN pair

# These three thresholds are round numbers taken from the ALREADY-PUBLISHED
# baseline description in DAILY_FIT_PROTOCOL.md (S = 88.29%, spread 14.94pp,
# ~98% within-basin vs ~63% across-basin). They are NOT read off this run's
# numbers, and they are NOT moved afterwards.

PREDICTIONS = [
 ('P1', 'S rises to > %.0f%% AND the pairwise matrix stops being bimodal' % P1_S_MIN),
 ('P2', 'the two-basin structure (~98% within / ~63% across) disappears or weakens'),
 ('P3', 'L does NOT improve much and may worsen -- EXPECTED, not a failure'),
 ('P4', 'occupancy stabilises across seed sets'),
]
print('=' * 96)
print('PRE-REGISTERED PREDICTIONS  (frozen in DAILY_FIT_PROTOCOL.md before any number existed)')
print('=' * 96)
for k, t in PREDICTIONS:
    print(f'  {k}: {t}')
print('-' * 96)
print('GRADING ARITHMETIC, declared now:')
print(f'  BIMODAL(matrix) := (max-min of the 6 pairwise values) >= {BIMODAL_SPREAD:.1f} pp')
print(f'                     AND largest gap between sorted values >= {BIMODAL_GAP:.1f} pp')
print(f'  P1 PASS := S > {P1_S_MIN:.1f}%  AND  not BIMODAL')
print(f'  P2 PASS := count of pairwise values < {BASIN_LOW:.1f}% is STRICTLY LOWER than the')
print('             baseline count (0 vs 0 is reported as UNTESTABLE, not as a pass)')
print('  P3      := reported as consistent/inconsistent. A WORSE L is EXPECTED and is')
print('             not counted against the architecture. Only a large IMPROVEMENT in L')
print('             would be inconsistent with P3.')
print('  P4 PASS := max across-seed-set occupancy spread (pp, over the 5 labels) is')
print("             LOWER than the baseline's")
print('-' * 96)
print("ARM 3's FEATURE SET, declared as a RULE before the correlation matrix is seen:")
print('  momentum slot   = TREND_FEATURE (the engine\'s own pre-existing choice)')
print('  volatility slot = vol_2h        (the engine\'s own primary volatility feature)')
print('  dispersion slot = whichever of {dist_ma, drawdown, vix_chg} has the LOWEST')
print('                    MAXIMUM |r| against the other two, on the DAILY matrix.')
print('  This is LABEL-BLIND: no label, no metric and no forward return enters it. It is')
print('  a rule, not a pick -- "decollinearized" has to mean decollinearized, and section')
print('  3 reports whether the chosen triple actually achieves it.')
print('-' * 96)
print('IF P1 AND P2 BOTH FAIL -> the small-sample diagnosis is WRONG. Section 8 prints')
print('that verdict without softening it.')
print('=' * 96)

assert BAR_DIR_WEIGHT == 0.0, 'BAR_DIR_WEIGHT must be 0.0 for this notebook'
W_FIXED = 0.0

# Metric parameters, reused VERBATIM from STABILITY_LAG_PROTOCOL.md.
R_SEED_SETS   = 4       # R = 4 DISJOINT seed sets. NOT reducible -- S is meaningless without it.
ZZ_PCT        = 2.0     # ZigZag reversal threshold for L
MACRO_ZZ_PCT  = 10.0    # ZigZag threshold used ONLY to COUNT macro regimes in a fit window
W_GUARD_MAX   = 25.0    # G3
OCC_MIN_PCT   = 3.0     # G1 / G2
OCC_MAX_PCT   = 50.0    # G1
SIDEWAYS_MAX  = 50.0    # G4
COLLINEAR_HI  = 0.70    # declared NOW: |r| >= this between two features is "high"

print(f'\nR={R_SEED_SETS} disjoint seed sets   L-ZigZag={ZZ_PCT}%   '
      f'macro-regime ZigZag={MACRO_ZZ_PCT}%   BAR_DIR_WEIGHT={W_FIXED} (asserted)')
print(f'guards: G1 occ in [{OCC_MIN_PCT},{OCC_MAX_PCT}]%  G2 no collapse  '
      f'G3 W<={W_GUARD_MAX}  G4 SIDEWAYS<={SIDEWAYS_MAX}%')
print(f'collinearity is called HIGH at |r| >= {COLLINEAR_HI} -- declared before the matrix '
      'is printed')

## 3. Data — daily ^NSEI (long history) and the 2h series

`^NSEI` daily reaches ~2007 on yfinance; India VIX starts later. **VIX is not
forward-filled across years in which it does not exist.** The fit window is
restricted to the span where *both* series exist, and the cost of that
restriction is printed. VIX is kept rather than dropped so that arm 2 really is
"the same nine features" — dropping it would confound the data change with a
feature change, which the protocol forbids.

`N_REGIMES_OBSERVED` — distinct macro price legs (a 10% ZigZag) inside each arm's
**fit window** — is the headline number. The entire argument rests on the daily
arms seeing materially more regimes than the ~6–10 the 2h fit sees.

In [ ]:
# ===========================================================================
# DAILY DATA. Real yfinance if reachable; otherwise a LONG synthetic daily
# series from which the 2h series is REBUILT so the two are mutually consistent
# (an inconsistent pair would make the causality asserts meaningless theatre).
# ===========================================================================
def load_daily():
    '''(daily_close, daily_vix, is_synth). Real ^NSEI period=max if reachable.'''
    try:
        import yfinance as yf

        def d1(tk):
            raw = yf.download(tk, interval='1d', period='max',
                              auto_adjust=True, progress=False)
            if raw is None or len(raw) == 0:
                return None
            c = raw['Close'].squeeze().dropna()
            idx = pd.to_datetime(c.index)
            c.index = (idx.tz_localize(None) if idx.tz is not None else idx).normalize()
            return c[~c.index.duplicated(keep='last')]

        nd = d1('^NSEI')
        if nd is None or len(nd) < 500:
            raise ValueError('too few daily bars')
        vd = d1('^INDIAVIX')
        if vd is None or len(vd) < 200:
            raise ValueError('no India VIX daily history')
        nd.name, vd.name = 'nifty_d', 'vix_d'
        print(f'yfinance daily: ^NSEI {len(nd)} bars {nd.index[0]:%Y-%m-%d} -> '
              f'{nd.index[-1]:%Y-%m-%d}')
        print(f'yfinance daily: ^INDIAVIX {len(vd)} bars {vd.index[0]:%Y-%m-%d} -> '
              f'{vd.index[-1]:%Y-%m-%d}')
        return nd, vd, False
    except Exception as e:
        print(f'daily yfinance unavailable ({e}); falling back to SYNTHETIC daily.')
        return _synth_daily()


def _synth_daily(years=18, seed=20260730):
    '''Regime-blocked GBM daily series with MANY macro regimes, plus an OU VIX
    that deliberately STARTS LATE, so the short-VIX-history handling is
    exercised rather than assumed.'''
    rng = np.random.default_rng(seed)
    days = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=int(years * 252))
    n = len(days)
    # 16 macro regimes, alternating in character, so N_REGIMES_OBSERVED on the
    # synthetic daily fit window is realistically LARGE.
    blocks = [(0.00050, 0.0075), (-0.00090, 0.0140), (0.00035, 0.0060),
              (0.00080, 0.0070), (-0.00160, 0.0210), (0.00060, 0.0090),
              (0.00020, 0.0055), (-0.00070, 0.0130), (0.00095, 0.0080),
              (0.00010, 0.0065), (-0.00120, 0.0175), (0.00070, 0.0085),
              (0.00040, 0.0060), (-0.00050, 0.0115), (0.00085, 0.0075),
              (0.00030, 0.0068)]
    edges = np.linspace(0, n, len(blocks) + 1).astype(int)
    lvl, out = 2500.0, []
    for bi, (mu, sg) in enumerate(blocks):
        for _ in range(edges[bi + 1] - edges[bi]):
            lvl *= np.exp(rng.normal(mu, sg))
            out.append(lvl)
    nd = pd.Series(out, index=days, name='nifty_d')
    # VIX starts LATE ON PURPOSE (2 years in), mimicking India VIX vs ^NSEI.
    vstart = int(2 * 252)
    vv, p = [], 15.0
    for i in range(vstart, n):
        tgt = 12.0 + 900.0 * blocks[min(int(i / n * len(blocks)), len(blocks) - 1)][1]
        p = max(p + 0.06 * (tgt - p) + rng.normal(0, 1.2), 8.0)
        vv.append(p)
    vd = pd.Series(vv, index=days[vstart:], name='vix_d')
    return nd, vd, True


def rebuild_2h_from_daily(nd, vd, n_days=730, seed=7, sigma_intra=0.35):
    '''Expand the LAST n_days daily bars into 3 x 2h bars/day by a Brownian
    bridge whose LAST bar of day d equals the daily close of day d.

    Used only on the synthetic path. It exists so the daily and 2h series in the
    synthetic run describe the SAME market -- otherwise the daily->2h projection
    would be projecting one random walk onto an unrelated one and every causality
    assert would pass while measuring nothing.
    '''
    rng = np.random.default_rng(seed)
    d = nd.iloc[-(n_days + 1):]
    lg = np.log(d.values)
    idx, val = [], []
    for i in range(1, len(d)):
        lp, lc = lg[i - 1], lg[i]
        step = (lc - lp)
        for k, h in enumerate((10, 12, 14), start=1):
            base = lp + step * k / 3.0
            noise = 0.0 if k == 3 else rng.normal(
                0, abs(step) * sigma_intra + 1e-4) * np.sqrt(k * (3 - k) / 3.0)
            idx.append(d.index[i] + pd.Timedelta(hours=h))
            val.append(float(np.exp(base + noise)))
    px = pd.Series(val, index=pd.DatetimeIndex(idx), name='nifty')
    vx = (vd.reindex(d.index).ffill().bfill()
            .reindex(pd.DatetimeIndex(idx).normalize()).values)
    vx = pd.Series(vx * (1.0 + rng.normal(0, 0.01, len(idx))),
                   index=px.index, name='vix')
    return px, vx


NIFTY_D, VIX_D, DAILY_SYNTH = load_daily()

if DAILY_SYNTH:
    # Consistency decree: if the daily series is synthetic, the 2h series is
    # REBUILT from it. A real 2h series paired with a synthetic daily one would
    # make every projection assert vacuous.
    nifty, vix = rebuild_2h_from_daily(NIFTY_D, VIX_D)
    TAC_SYNTH = True
    print('\nSYNTHETIC daily -> the 2h series has been REBUILT from it so that the two '
          'describe the same market.')

if TAC_SYNTH or DAILY_SYNTH:
    print('#' * 100)
    for _ in range(3):
        print('#   SYNTHETIC - ILLUSTRATIVE ONLY   ' * 2)
    print('#' * 100)
    print('#  yfinance is firewalled in this sandbox. Every number and chart below comes')
    print('#  from a synthetic series. This run is a PLUMBING TEST of the daily-fit')
    print('#  machinery and the causality proofs ONLY. It says NOTHING about real Nifty')
    print('#  regimes, real fit stability, or whether the predictions hold. The REAL')
    print('#  verdict requires the Kaggle run, where yfinance works.')
    print('#' * 100)
else:
    print('\n' + '=' * 100)
    print('REAL yfinance data on BOTH cadences. Verdicts below are decision-grade.')
    print('=' * 100)

# ---- THE VIX SPAN PROBLEM, handled explicitly -----------------------------
_d0_px, _d0_vx = NIFTY_D.index[0], VIX_D.index[0]
DAILY_START = max(_d0_px, _d0_vx)
NIFTY_D = NIFTY_D[NIFTY_D.index >= DAILY_START]
VIX_D = VIX_D.reindex(NIFTY_D.index).ffill()
assert VIX_D.notna().all(), 'VIX has holes INSIDE its own span -- investigate before using'
_lost_yrs = (DAILY_START - _d0_px).days / 365.25
print(f'\nVIX SPAN HANDLING (explicit, per protocol)')
print(f'  ^NSEI daily starts     : {_d0_px:%Y-%m-%d}')
print(f'  India VIX daily starts : {_d0_vx:%Y-%m-%d}')
print(f'  DECISION: RESTRICT the daily window to {DAILY_START:%Y-%m-%d}, the first date '
      f'both series exist.')
print(f'  WHY: keeping vix_chg makes arm 2 genuinely "the SAME nine features", so the')
print(f'       data change is not confounded with a feature change. VIX is NOT')
print(f'       back-filled across years it does not exist -- those years are DROPPED.')
print(f'  COST: {_lost_yrs:.1f} years of price history discarded '
      f'({len(NIFTY_D)} daily bars retained).')
print(f'  (ffill is used only INSIDE the common span, for NSE sessions with no VIX '
      f'print; that is a same-day hole, not a missing era.)')

## 4. The daily→2h projection — the causality proof

Two independent proofs, both runtime asserts:

1. **Per-bar assert.** Every 2h bar's source daily timestamp is *strictly* earlier
   than that bar's own session date.
2. **Truncation probe.** The whole pipeline — features, scaler, HMM fit,
   projection, labelling — is re-run on a series cut at `T` (with the daily series
   cut to sessions strictly before `T`'s session, i.e. exactly what a live engine
   would have had) and every label at `t ≤ T` must come out **bit-identical**.
   The assert alone is not the test; this is.

The daily fit is **anchored**: it is fit on the leading `TRAIN_FRACTION` of the
daily history and applied forward. It is never fit on all data and applied
backwards.

In [ ]:
# ===========================================================================
# FEATURES ON BOTH CADENCES.
#
# The DAILY features use `build_features` UNMODIFIED, with BASE_WIN unchanged.
# That means the window lengths are IDENTICAL IN BARS on both cadences
# (1 / 3 / 9 / 15 / 10 / 5 / 20 / 20), which is the cleanest reading of "the same
# features expressed as daily equivalents": same function, same geometry, same
# number of observations behind every feature.
#
# The consequence, stated plainly rather than buried: on daily bars those windows
# span 3x more CALENDAR time (mom_3d is a 9-DAY momentum, not a 9-BAR one). That
# is inherent to "learn slow" -- it is the change, not a side effect of it.
#
# The alternative -- matching calendar horizons -- would make mom_1d IDENTICAL to
# ret_2h on daily bars (both 1-day returns), a perfectly collinear duplicate
# column. That is a worse experiment, so it was not used.
# ===========================================================================
FEAT_2H = build_features(nifty, vix, LOOKBACK_SCALE)
FEAT_D = build_features(NIFTY_D, VIX_D, LOOKBACK_SCALE)

FEATURES_9 = list(FEATURE_COLS)
# Arm 3's slots, per the rule declared in section 2. The dispersion/context slot
# is RESOLVED in the next cell, from the daily correlation matrix only.
F3_MOM, F3_VOL = TREND_FEATURE, 'vol_2h'
F3_DISP_CANDIDATES = ['dist_ma', 'drawdown', 'vix_chg']

BASE_CFG = [c for c in CONFIGS if c['name'] == ADOPTED_CONFIG_NAME][0]
BASE_N, BASE_COV = BASE_CFG['N'], BASE_CFG['cov']
assert list(BASE_CFG['features']) == FEATURES_9, \
    'the adopted config is expected to carry all 9 features'

print(f'2h feature bars   : {len(FEAT_2H)}   {FEAT_2H.index[0]:%Y-%m-%d} -> '
      f'{FEAT_2H.index[-1]:%Y-%m-%d}')
print(f'daily feature bars: {len(FEAT_D)}   {FEAT_D.index[0]:%Y-%m-%d} -> '
      f'{FEAT_D.index[-1]:%Y-%m-%d}')
print(f'window lengths IN BARS (identical on both cadences): {BASE_WIN}')
print(f'model            : N={BASE_N} states, cov={BASE_COV!r} (the adopted config, '
      f'unchanged in every arm)')
print(f'params  9 feats  : {int(n_params(BASE_N, 9, BASE_COV))}   '
      f'3 feats: {int(n_params(BASE_N, 3, BASE_COV))}')

In [ ]:
# ===========================================================================
# THE FEATURE CORRELATION MATRICES -- the collinearity claim is CHECKED here,
# not assumed. If the correlations are LOW, the "nested momenta create likelihood
# ridges" explanation for the bimodality is WRONG, and this cell says so.
# ===========================================================================
def corr_report(df, feats, tag):
    C = df[feats].corr()
    print(f'\n{tag}   ({len(df)} bars, {len(feats)} features)')
    print(C.round(2).to_string())
    A = C.abs().values.copy()
    np.fill_diagonal(A, 0.0)
    _cn = np.linalg.cond(C.values)
    print(f'  max |r| off-diagonal = {A.max():.3f}   '
          f'pairs with |r| >= {COLLINEAR_HI}: '
          f'{int((np.triu(A, 1) >= COLLINEAR_HI).sum())} of '
          f'{len(feats) * (len(feats) - 1) // 2}')
    print(f'  condition number of the correlation matrix = {_cn:,.1f}   '
          '(a likelihood-ridge proxy: large = flat directions in the objective)')
    return C, float(A.max()), float(_cn)


MOM = ['mom_1d', 'mom_3d', 'mom_5d']
C2H, MX2H, CN2H = corr_report(FEAT_2H, FEATURES_9, 'A. 2h, 9 features  [BASELINE fit input]')
CD9, MXD9, CND9 = corr_report(FEAT_D, FEATURES_9, 'B. DAILY, 9 features  [arm 2 fit input]')

# ---- RESOLVE arm 3's dispersion slot by the rule declared in section 2 ------
print(f'\nARM 3 dispersion/context slot -- applying the pre-declared rule '
      f'(momentum={F3_MOM}, volatility={F3_VOL}):')
_sc = {d: max(abs(CD9.loc[d, F3_MOM]), abs(CD9.loc[d, F3_VOL]))
       for d in F3_DISP_CANDIDATES}
for d, v in _sc.items():
    print(f'    {d:<9} max |r| against the other two slots = {v:.3f}')
F3_DISP = min(_sc, key=_sc.get)
FEATURES_3 = [F3_MOM, F3_VOL, F3_DISP]
assert all(f in FEATURES_9 for f in FEATURES_3)
print(f'  -> dispersion slot = {F3_DISP!r}')
if F3_DISP != 'dist_ma':
    print(f'  NOTE: the "obvious" role-based pick, dist_ma, scores {_sc["dist_ma"]:.3f} and is')
    print('        REJECTED by the rule. Hard-coding it would have produced an arm labelled')
    print('        "decollinearized" that was not -- which is exactly the assumption this')
    print('        section exists to stop.')

CD3, MXD3, CND3 = corr_report(FEAT_D, FEATURES_3, 'C. DAILY, 3 features  [arm 3 fit input]')
if MXD3 >= COLLINEAR_HI:
    print(f'  *** ARM 3 IS NOT ACTUALLY DECOLLINEARIZED: max |r| = {MXD3:.3f} >= '
          f'{COLLINEAR_HI}. ***')
    print('  Whatever arm 3 measures, it is NOT "the collinearity removed". Read its row')
    print('  with that in mind.')
else:
    print(f'  arm 3 IS decollinearized: max |r| = {MXD3:.3f} < {COLLINEAR_HI}, condition '
          f'number {CND3:.1f} vs {CND9:.1f} for the 9-feature daily set.')

print('\n' + '=' * 96)
print('THE COLLINEARITY CLAIM, CHECKED')
print('=' * 96)
for _tag, _C in (('2h   ', C2H), ('daily', CD9)):
    _tri = [(a, b, abs(_C.loc[a, b])) for a, b in itertools.combinations(MOM, 2)]
    print(f'  {_tag} nested momentum trio: ' +
          '   '.join(f'|r({a},{b})| = {v:.3f}' for a, b, v in _tri))
_mx2 = max(abs(C2H.loc[a, b]) for a, b in itertools.combinations(MOM, 2))
_mxd = max(abs(CD9.loc[a, b]) for a, b in itertools.combinations(MOM, 2))
print('-' * 96)
if _mx2 >= COLLINEAR_HI:
    print(f'  VERDICT: the 2h momentum trio IS highly collinear (max |r| = {_mx2:.3f} '
          f'>= {COLLINEAR_HI}).')
    print('  The likelihood-ridge explanation for the bimodality is CONSISTENT with the')
    print('  data. Consistent is not proven -- collinearity is necessary for that story,')
    print('  not sufficient. Arm 3 is the actual test of it.')
else:
    print(f'  VERDICT: the 2h momentum trio is NOT highly collinear '
          f'(max |r| = {_mx2:.3f} < {COLLINEAR_HI}).')
    print('  *** THE COLLINEARITY EXPLANATION FOR THE BIMODALITY IS THEREFORE WRONG. ***')
    print('  The nested mom_1d/mom_3d/mom_5d features are not creating a likelihood')
    print('  ridge here, so whatever produces the two basins is something else. This is')
    print('  reported as a refutation, not softened.')
print(f'  (daily trio max |r| = {_mxd:.3f}; condition number 2h {CN2H:,.0f} vs '
      f'daily {CND9:,.0f} vs daily-3feat {CND3:,.0f})')
print('=' * 96)

In [ ]:
# ===========================================================================
# THE CAUSAL DAILY -> 2h PROJECTION.
# ===========================================================================
def daily_source_index(daily_index, bar_index):
    '''For every bar in `bar_index`, the position in `daily_index` of the last
    daily bar STRICTLY BEFORE that bar's session date.

    searchsorted(..., 'left') - 1 gives the last daily date < session date, so a
    bar on day d can only ever see day d-1 or earlier. Returns -1 where no such
    daily bar exists (those bars are dropped, loudly).
    '''
    sess = pd.DatetimeIndex(bar_index).normalize().values
    di = pd.DatetimeIndex(daily_index).normalize().values
    assert (np.diff(di.astype('int64')) > 0).all(), 'daily index must be strictly increasing'
    return np.searchsorted(di, sess, side='left') - 1, sess, di


SRC_IDX, _SESS, _DI = daily_source_index(FEAT_D.index, FEAT_2H.index)
_have = SRC_IDX >= 0
if not _have.all():
    print(f'dropping {int((~_have).sum())} leading 2h bars with no fully-closed daily '
          f'bar behind them')

# EVERY arm is restricted to the SAME bar set, so S / L / W stay comparable.
FEAT_DF = FEAT_2H[_have]
DATES = FEAT_DF.index
SRC_IDX = SRC_IDX[_have]
CLOSE = nifty.reindex(DATES).values
TREND_RAW = FEAT_DF[TREND_FEATURE].values
N_BARS = len(DATES)
N_FIT = max(int(N_BARS * TRAIN_FRACTION), 50)          # 2h anchored fit window
N_FIT_D = max(int(len(FEAT_D) * TRAIN_FRACTION), 50)   # DAILY anchored fit window

# ---- PROOF 1: the per-bar causality assert --------------------------------
_src_ts = _DI[SRC_IDX]
_bar_sess = pd.DatetimeIndex(DATES).normalize().values
assert (_src_ts < _bar_sess).all(), \
    'CAUSALITY VIOLATION: a 2h bar is reading a daily bar from its own session or later'
_gap_days = (_bar_sess - _src_ts).astype('timedelta64[D]').astype(int)
print('ASSERT OK (proof 1): for all %d 2h bars, source daily date < bar session date.' % N_BARS)
print(f'  source-to-bar gap in calendar days: min {_gap_days.min()}  median '
      f'{int(np.median(_gap_days))}  max {_gap_days.max()}  '
      '(min must be >= 1, and it is)')
assert _gap_days.min() >= 1

print(f'\n2h  : {N_BARS} bars   {DATES[0]:%Y-%m-%d} -> {DATES[-1]:%Y-%m-%d}   '
      f'anchored fit window = {N_FIT} bars')
print(f'daily: {len(FEAT_D)} bars   {FEAT_D.index[0]:%Y-%m-%d} -> '
      f'{FEAT_D.index[-1]:%Y-%m-%d}   anchored fit window = {N_FIT_D} bars '
      f'({FEAT_D.index[0]:%Y-%m-%d} -> {FEAT_D.index[N_FIT_D - 1]:%Y-%m-%d})')
if FEAT_D.index[N_FIT_D - 1] < DATES[0]:
    print('  NOTE: the DAILY fit window CLOSES BEFORE the 2h evaluation span even begins,')
    print('        so arms 2 and 3 are scored entirely out of sample on the daily fit.')

In [ ]:
# ===========================================================================
# THE DAILY DIRECTION MODEL + its projection onto 2h bars.
# ===========================================================================
def seed_sets(K, R=R_SEED_SETS, base=BASE_SEED):
    ss = [list(range(base + r * K, base + r * K + K)) for r in range(R)]
    flat = [s for st in ss for s in st]
    assert len(set(flat)) == len(flat), 'seed sets must be DISJOINT'
    return ss


def scaled(df, feats, n_fit):
    raw = df[list(feats)].values
    return StandardScaler().fit(raw[:n_fit]).transform(raw)


def daily_masses(feat_d, feats, n_fit_d, seeds, N=None, cov=None):
    '''ANCHORED daily fit -> causal filtered direction masses per DAILY bar.

    The models see ONLY feat_d[:n_fit_d]; the masses come from the forward-only
    filtered posteriors, so daily bar j uses daily bars <= j and nothing later.
    '''
    N = BASE_N if N is None else N
    cov = BASE_COV if cov is None else cov
    Xd = scaled(feat_d, feats, n_fit_d)
    models, _ = fit_hmm_ensemble(Xd[:n_fit_d], N, cov, K=len(seeds), base_seed=seeds[0])
    b, s, r, probs = ensemble_direction_masses_by_mode(
        models, Xd, list(feats), DIRECTION_EXCLUDE, DIRECTION_MODE, None, None, None)
    return models, np.stack([b, s, r], axis=1), probs


def project(mass_d, probs_d, src_idx):
    '''Daily -> 2h. Row t of the output is the daily row for the last FULLY
    CLOSED session before bar t. Because `src_idx` is constant within a calendar
    day, this is exactly "shift one full day, then forward-fill".'''
    return mass_d[src_idx], probs_d[src_idx]


@contextlib.contextmanager
def projected_direction(mass_2h, probs_2h):
    '''Route label_bars' direction masses through the PROJECTED DAILY masses.

    label_bars resolves `ensemble_direction_masses_by_mode` from the notebook
    globals at call time, so rebinding that name substitutes the direction axis
    WITHOUT editing a line of the inlined engine. The INTENSITY axis
    (trend_raw / efficiency / gate_band) is untouched and stays on 2h -- that is
    the whole architecture in one context manager.
    '''
    g = globals()
    old = g['ensemble_direction_masses_by_mode']

    def _proj(models, Xs, feature_subset, exclude=None, direction_mode=None,
              tau=None, c=None, scale=None):
        assert len(Xs) == len(mass_2h), 'projected masses are misaligned with the bars'
        return mass_2h[:, 0], mass_2h[:, 1], mass_2h[:, 2], probs_2h

    g['ensemble_direction_masses_by_mode'] = _proj
    try:
        yield
    finally:
        g['ensemble_direction_masses_by_mode'] = old


print('projection machinery ready: daily fit -> filtered daily masses -> shift one '
      'full session -> forward-fill onto 2h bars -> injected as the DIRECTION axis.')
print('The INTENSITY axis (trend_z, efficiency, the H/L gate) remains 100% 2h in every arm.')

In [ ]:
# ===========================================================================
# PROOF 2 -- THE TRUNCATION PROBE. The real test.
#
# Re-run the ENTIRE pipeline on a series cut at T, with the daily series cut to
# sessions STRICTLY BEFORE T's session -- exactly the information a live engine
# would have held at T. Every label at t <= T must be BIT-IDENTICAL.
#
# The anchored fit windows are held at their ABSOLUTE bar counts, because that is
# what "anchored" means: the model is fit on a fixed leading window and applied
# forward. A probe that also shrank the fit window would be testing a different
# model, not the causality of this one.
# ===========================================================================
def run_arm(px, vx, feat_d_px, feat_d_vx, feats, seeds, n_fit_2h, n_fit_d,
            daily=True, end=None):
    '''One full end-to-end labelling. `end` truncates the 2h series at that
    timestamp and the daily series at the last session strictly before it.'''
    f2 = build_features(px, vx, LOOKBACK_SCALE)
    fd = build_features(feat_d_px, feat_d_vx, LOOKBACK_SCALE)
    if end is not None:
        f2 = f2[f2.index <= end]
        fd = fd[fd.index < pd.Timestamp(end).normalize()]
    src, _, di = daily_source_index(fd.index, f2.index)
    keep = src >= 0
    f2, src = f2[keep], src[keep]
    dates = f2.index
    assert (di[src] < pd.DatetimeIndex(dates).normalize().values).all(), \
        'CAUSALITY VIOLATION inside run_arm'
    X2 = scaled(f2, FEATURES_9, n_fit_2h)
    tr = f2[TREND_FEATURE].values
    if daily:
        dmodels, md_, pd_ = daily_masses(fd, feats, n_fit_d, seeds)
        m2, p2 = project(md_, pd_, src)
        with projected_direction(m2, p2):
            out = label_bars(dmodels, X2, dates, px, tr, n_fit_2h, FEATURES_9,
                             dir_feats=f2)
    else:
        Xf = scaled(f2, feats, n_fit_2h)
        models, _ = fit_hmm_ensemble(Xf[:n_fit_2h], BASE_N, BASE_COV,
                                     K=len(seeds), base_seed=seeds[0])
        out = label_bars(models, Xf, dates, px, tr, n_fit_2h, list(feats),
                         dir_feats=f2)
    return dates, out['tactical_regime_state'].values.copy()


_pseeds = seed_sets(ENSEMBLE_K)[0]
_t0 = time.time()
_d_full, _l_full = run_arm(nifty, vix, NIFTY_D, VIX_D, FEATURES_9, _pseeds,
                           N_FIT, N_FIT_D, daily=True)
_T = _d_full[-40]
_d_cut, _l_cut = run_arm(nifty[nifty.index <= _T], vix[vix.index <= _T],
                         NIFTY_D, VIX_D, FEATURES_9, _pseeds,
                         N_FIT, N_FIT_D, daily=True, end=_T)

_common = _d_full <= _T
assert len(_l_cut) == int(_common.sum()), \
    f'truncation changed the BAR COUNT: {len(_l_cut)} vs {int(_common.sum())}'
assert (_d_full[_common] == _d_cut).all(), 'truncation changed the bar TIMESTAMPS'
_bad = int((_l_full[_common] != _l_cut).sum())
print(f'TRUNCATION PROBE   cut at {_T:%Y-%m-%d %H:%M}   '
      f'{int(_common.sum())} labels compared   mismatches = {_bad}   '
      f'({time.time() - _t0:.1f}s)')
assert _bad == 0, (
    f'*** HARD STOP: {_bad} labels changed when the series was truncated. The '
    'daily->2h projection is NOT causal. Do not work around this. ***')
print('ASSERT OK (proof 2): truncating the series left every label at t <= T '
      'BIT-IDENTICAL.')
print('The daily->2h projection is CAUSAL by measurement, not by claim.')

## 5. Metrics S / L / W, guards G1–G4, and the ZigZag leakage tripwire

Reused **verbatim** from `stability_lag.ipynb` so rows are comparable across the
two notebooks. The ZigZag deliberately uses future prices — it is a
*retrospective* grading device — so it is fenced off from labelling by three
runtime asserts, and `G4` is driven by a degenerate all-`SIDEWAYS` stub and
asserted to void it.

In [ ]:
# ===========================================================================
# THE ZIGZAG -- LABEL-BLIND, RETROSPECTIVE. VERBATIM from stability_lag.ipynb.
# ===========================================================================
ZZ_CALLS = 0
ZZ_IDS = set()
ZZ_ARRAYS = []


def zigzag_pivots(px, pct=ZZ_PCT):
    global ZZ_CALLS
    px = np.asarray(px, dtype=float)
    n = len(px)
    if n < 3:
        out = np.array([], dtype=int)
    else:
        hi_i = lo_i = 0
        d, start, piv, ext_i = 0, None, [], 0
        for i in range(1, n):
            if px[i] > px[hi_i]:
                hi_i = i
            if px[i] < px[lo_i]:
                lo_i = i
            if (px[hi_i] - px[i]) / px[hi_i] * 100.0 >= pct:
                d, piv, ext_i, start = -1, [hi_i], i, i
                break
            if (px[i] - px[lo_i]) / px[lo_i] * 100.0 >= pct:
                d, piv, ext_i, start = +1, [lo_i], i, i
                break
        if d == 0:
            out = np.array([], dtype=int)
        else:
            for i in range(start + 1, n):
                if d > 0:
                    if px[i] >= px[ext_i]:
                        ext_i = i
                    elif (px[ext_i] - px[i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = -1, i
                else:
                    if px[i] <= px[ext_i]:
                        ext_i = i
                    elif (px[i] - px[ext_i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = +1, i
            if ext_i != piv[-1]:
                piv.append(ext_i)
            out = np.asarray(piv, dtype=int)
    ZZ_CALLS += 1
    ZZ_IDS.add(id(out))
    ZZ_ARRAYS.append(out)
    return out


def zigzag_swings(px, pct=ZZ_PCT):
    px = np.asarray(px, dtype=float)
    piv = zigzag_pivots(px, pct)
    sw = []
    for a, b in zip(piv[:-1], piv[1:]):
        move = (px[b] - px[a]) / px[a] * 100.0
        if abs(move) >= pct:
            sw.append((int(a), int(b), 1 if move > 0 else -1))
    ZZ_IDS.add(id(sw))
    return sw


# ---- the leakage tripwire -------------------------------------------------
_label_bars_raw = label_bars
LABEL_CALLS = 0
LABEL_PHASE_OPEN = True


def label_bars(*args, **kwargs):
    global LABEL_CALLS
    assert ZZ_CALLS == 0 or LABEL_PHASE_OPEN, (
        'LEAKAGE: a label was produced AFTER a ZigZag had been computed.')
    _bw = kwargs.get('bar_dir_weight', BAR_DIR_WEIGHT)
    assert _bw == 0.0, f'BAR_DIR_WEIGHT drifted to {_bw} inside a labelling call'
    for v in list(args) + list(kwargs.values()):
        assert id(v) not in ZZ_IDS, 'LEAKAGE: a ZigZag output was passed to label_bars'
        if isinstance(v, np.ndarray):
            for z in ZZ_ARRAYS:
                assert not np.shares_memory(v, z), \
                    'LEAKAGE: a label_bars argument aliases ZigZag memory'
    LABEL_CALLS += 1
    return _label_bars_raw(*args, **kwargs)


DIR_OF_LABEL = {'H_BULL': 1, 'L_BULL': 1, 'SIDEWAYS': 0, 'L_BEAR': -1, 'H_BEAR': -1}


def emitted_direction(labels):
    return np.array([DIR_OF_LABEL[x] for x in np.asarray(labels)], dtype=int)


def metric_S(label_sets):
    R = len(label_sets)
    M = np.full((R, R), np.nan)
    vals = []
    for i in range(R):
        M[i, i] = 100.0
        for j in range(i + 1, R):
            a, b = np.asarray(label_sets[i]), np.asarray(label_sets[j])
            assert len(a) == len(b)
            v = 100.0 * float(np.mean(a == b))
            M[i, j] = M[j, i] = v
            vals.append(v)
    return float(np.mean(vals)), M, vals


def metric_L(labels, swings):
    d = emitted_direction(labels)
    lags, unmatched = [], 0
    for i0, i1, sgn in swings:
        seg = d[i0:i1 + 1]
        hit = np.flatnonzero(seg == sgn)
        if hit.size:
            lags.append(int(hit[0]))
        else:
            lags.append(int(i1 - i0)); unmatched += 1
    if not lags:
        return np.nan, np.nan, np.array([]), 0
    a = np.asarray(lags, dtype=float)
    return float(np.median(a)), float(np.percentile(a, 75)), a, unmatched


def metric_W(labels):
    a = np.asarray(labels)
    return 0.0 if len(a) < 2 else 100.0 * float(np.sum(a[1:] != a[:-1])) / (len(a) - 1)


def occupancy_pct(labels):
    a = np.asarray(labels)
    return {lb: 100.0 * float(np.mean(a == lb)) for lb in REGIME_LABELS}


def evaluate_guards(occ, W):
    g = {}
    bad_hi = [l for l in REGIME_LABELS if occ.get(l, 0.0) > OCC_MAX_PCT]
    bad_lo = [l for l in REGIME_LABELS if occ.get(l, 0.0) < OCC_MIN_PCT]
    g['G1'] = (not bad_hi and not bad_lo, 'ok' if not (bad_hi or bad_lo)
               else f'outside [{OCC_MIN_PCT},{OCC_MAX_PCT}]%: '
                    + ','.join(sorted(set(bad_hi + bad_lo))))
    g['G2'] = (not bad_lo, 'ok' if not bad_lo else 'collapsed: ' + ','.join(bad_lo))
    g['G3'] = (W <= W_GUARD_MAX, f'W={W:.2f} vs max {W_GUARD_MAX}')
    g['G4'] = (occ.get('SIDEWAYS', 0.0) <= SIDEWAYS_MAX,
               f"SIDEWAYS={occ.get('SIDEWAYS', 0.0):.1f}% vs max {SIDEWAYS_MAX}%")
    return g


def guards_pass(g):
    return all(v[0] for v in g.values())


# ---- ZigZag unit test on a toy array, then the bookkeeping is RESET so the
# ---- tripwire starts the label phase from a true zero.
_saw = np.array([100, 101, 100.5, 105, 104, 101, 100, 103, 108, 107, 102, 106.])
_sw = zigzag_swings(_saw, pct=2.0)
assert len(_sw) >= 3
for _a, _b, _s in _sw:
    _mv = (_saw[_b] - _saw[_a]) / _saw[_a] * 100
    assert _a < _b and np.sign(_mv) == _s and abs(_mv) >= 2.0
ZZ_CALLS = 0; ZZ_IDS.clear(); ZZ_ARRAYS.clear()
print(f'ZigZag unit test PASS ({len(_sw)} alternating swings); bookkeeping reset, '
      'tripwire starts from ZZ_CALLS = 0.')
print('metrics ready: metric_S (full pairwise matrix), metric_L, metric_W, '
      'evaluate_guards G1-G4. NO composite score is defined anywhere.')

## 6. Label phase — three arms × R = 4 disjoint seed sets

Everything is labelled **before any ZigZag exists**; that ordering is the leakage
proof. Across the three arms only the **direction** axis changes. The intensity
gate, the confirmation delay, the hysteresis, `CONF_L`, the config `N`/`cov` and
the 2h price series are identical in all three — one knob at a time.

In [ ]:
# ===========================================================================
# THE LABEL PHASE.
# ===========================================================================
assert ZZ_CALLS == 0, 'a ZigZag was computed before the label phase'

ARMS = [
    dict(name='1 BASELINE (2h fit)', daily=False, features=FEATURES_9,
         note='current 2h fit, existing 9 features'),
    dict(name='2 DAILY-9FEAT', daily=True, features=FEATURES_9,
         note='daily fit, long history, SAME 9 features -- isolates the DATA change'),
    dict(name='3 DAILY-3FEAT', daily=True, features=FEATURES_3,
         note='daily fit + decollinearized features -- adds the FEATURE change'),
]

X2_9 = scaled(FEAT_DF, FEATURES_9, N_FIT)
SEEDSETS = seed_sets(ENSEMBLE_K)
LABELS, FIT_COUNT, _t0 = {}, 0, time.time()

for a in ARMS:
    outs = []
    for sd in SEEDSETS:
        if a['daily']:
            dmodels, md_, pd_ = daily_masses(FEAT_D, a['features'], N_FIT_D, sd)
            m2, p2 = project(md_, pd_, SRC_IDX)
            with projected_direction(m2, p2):
                r = label_bars(dmodels, X2_9, DATES, nifty, TREND_RAW, N_FIT,
                               FEATURES_9, dir_feats=FEAT_DF)
        else:
            Xf = scaled(FEAT_DF, a['features'], N_FIT)
            models, _ = fit_hmm_ensemble(Xf[:N_FIT], BASE_N, BASE_COV,
                                         K=ENSEMBLE_K, base_seed=sd[0])
            r = label_bars(models, Xf, DATES, nifty, TREND_RAW, N_FIT,
                           list(a['features']), dir_feats=FEAT_DF)
        FIT_COUNT += ENSEMBLE_K
        outs.append(r['tactical_regime_state'].values.copy())
    LABELS[a['name']] = outs
    print(f"  labelled {a['name']:<22} feats={len(a['features'])}  "
          f"({time.time() - _t0:5.1f}s cumulative)")

print(f'\nLABEL PHASE COMPLETE in {time.time() - _t0:.1f}s   {FIT_COUNT} HMM fits, '
      f'{LABEL_CALLS} label_bars calls')
assert ZZ_CALLS == 0, 'LEAKAGE: a ZigZag existed during the label phase'
print('ASSERT OK: ZZ_CALLS == 0 -- every label was produced before any ZigZag existed.')
LABEL_PHASE_OPEN = False
print('LABEL PHASE CLOSED. Any label_bars call from here on fails the tripwire.')

In [ ]:
# ===========================================================================
# EVAL PHASE. ZigZags may now be computed.
# ===========================================================================
SWINGS = zigzag_swings(CLOSE, ZZ_PCT)
print(f'ZigZag ({ZZ_PCT}%) found {len(SWINGS)} swings over {N_BARS} 2h bars '
      f'(the L denominator).')


def n_regimes_observed(close, pct=MACRO_ZZ_PCT):
    '''Distinct macro price legs inside a window. This is a DESCRIPTIVE count of
    the fit window, computed in the EVAL phase; it never touches a label.'''
    return len(zigzag_swings(np.asarray(close, dtype=float), pct))


NREG_2H = n_regimes_observed(CLOSE[:N_FIT])
NREG_D = n_regimes_observed(NIFTY_D.reindex(FEAT_D.index).values[:N_FIT_D])
_fit_yrs_2h = (DATES[N_FIT - 1] - DATES[0]).days / 365.25
_fit_yrs_d = (FEAT_D.index[N_FIT_D - 1] - FEAT_D.index[0]).days / 365.25

print('\n' + '=' * 96)
print(f'N_REGIMES_OBSERVED  -- distinct {MACRO_ZZ_PCT:.0f}% macro legs inside each '
      f'arm\'s FIT WINDOW')
print('=' * 96)
print(f'  arm 1 BASELINE   2h fit window : {N_FIT:>6} bars  {_fit_yrs_2h:5.2f} yrs  '
      f'{DATES[0]:%Y-%m-%d} -> {DATES[N_FIT-1]:%Y-%m-%d}   '
      f'N_REGIMES_OBSERVED = {NREG_2H}')
print(f'  arms 2 & 3       daily fit win : {N_FIT_D:>6} bars  {_fit_yrs_d:5.2f} yrs  '
      f'{FEAT_D.index[0]:%Y-%m-%d} -> {FEAT_D.index[N_FIT_D-1]:%Y-%m-%d}   '
      f'N_REGIMES_OBSERVED = {NREG_D}')
print('-' * 96)
print(f'  rows per free parameter   2h: {N_FIT / n_params(BASE_N, 9, BASE_COV):.1f}   '
      f'daily 9-feat: {N_FIT_D / n_params(BASE_N, 9, BASE_COV):.1f}   '
      f'daily 3-feat: {N_FIT_D / n_params(BASE_N, 3, BASE_COV):.1f}')
print(f'  REGIME MULTIPLE: the daily fit sees {NREG_D / max(NREG_2H, 1):.1f}x as many '
      f'macro legs as the 2h fit.')
if NREG_D <= NREG_2H * 1.5:
    print('  *** THE PREMISE OF THIS NOTEBOOK IS NOT MET ON THIS DATA: the daily fit')
    print('      window does NOT contain materially more macro regimes. Every result')
    print('      below is therefore testing something weaker than the intended change.')
print('=' * 96)

## 7. Results — S, L and W together, on every row. Baseline is row 1.

The **pairwise S matrix is printed per arm**: the bimodality P1 and P2 are about
is only visible there, never in the mean.

In [ ]:
# ===========================================================================
# SUMMARISE EVERY ARM, then DRIVE G4 with a degenerate stub through the SAME
# code path, so the guard is tested rather than trusted.
# ===========================================================================
def summarise_arm(name, label_sets, note='', rows_per_param=np.nan, nreg=np.nan):
    S, M, pair_vals = metric_S(label_sets)
    L_med = [metric_L(lb, SWINGS)[0] for lb in label_sets]
    L_p75 = [metric_L(lb, SWINGS)[1] for lb in label_sets]
    unm = [metric_L(lb, SWINGS)[3] for lb in label_sets]
    Ws = [metric_W(lb) for lb in label_sets]
    occs = [occupancy_pct(lb) for lb in label_sets]
    occ = {lb: float(np.mean([o[lb] for o in occs])) for lb in REGIME_LABELS}
    occ_spread = max(max(o[lb] for o in occs) - min(o[lb] for o in occs)
                     for lb in REGIME_LABELS)
    W = float(np.mean(Ws))
    g = evaluate_guards(occ, W)
    sv = sorted(pair_vals)
    return dict(name=name, note=note, S=S, S_min=min(pair_vals), S_max=max(pair_vals),
                S_matrix=M, S_pairs=pair_vals,
                S_spread=max(pair_vals) - min(pair_vals),
                S_gap=max(np.diff(sv)) if len(sv) > 1 else 0.0,
                n_low_pairs=int(sum(1 for v in pair_vals if v < BASIN_LOW)),
                L=float(np.mean(L_med)), L_lo=float(np.min(L_med)),
                L_hi=float(np.max(L_med)), L75=float(np.mean(L_p75)),
                unmatched=float(np.mean(unm)),
                W=W, W_lo=float(np.min(Ws)), W_hi=float(np.max(Ws)),
                occ=occ, occ_spread=occ_spread, guards=g, void=not guards_pass(g),
                rows_per_param=rows_per_param, nreg=nreg)


def _rpp(a):
    return (round(N_FIT_D / n_params(BASE_N, len(a['features']), BASE_COV), 1)
            if a['daily'] else
            round(N_FIT / n_params(BASE_N, len(a['features']), BASE_COV), 1))


RESULTS = [summarise_arm(a['name'], LABELS[a['name']], a['note'], _rpp(a),
                         NREG_D if a['daily'] else NREG_2H) for a in ARMS]
RES = {r['name']: r for r in RESULTS}
BASE = RESULTS[0]

TAB = pd.DataFrame([{
    'arm': r['name'], 'N_REG': r['nreg'], 'rows/par': r['rows_per_param'],
    'S %': round(r['S'], 2), 'S spread pp': round(r['S_spread'], 2),
    'L med': round(r['L'], 2), 'L p75': round(r['L75'], 2),
    'W /100': round(r['W'], 2), 'unmat': round(r['unmatched'], 1),
    'SIDE %': round(r['occ']['SIDEWAYS'], 1), 'occ spread pp': round(r['occ_spread'], 2),
    'G1': 'P' if r['guards']['G1'][0] else 'F', 'G2': 'P' if r['guards']['G2'][0] else 'F',
    'G3': 'P' if r['guards']['G3'][0] else 'F', 'G4': 'P' if r['guards']['G4'][0] else 'F',
    'verdict': 'VOID' if r['void'] else 'ok',
    'dS': round(r['S'] - BASE['S'], 2), 'dL': round(r['L'] - BASE['L'], 2),
    'dW': round(r['W'] - BASE['W'], 2),
} for r in RESULTS])
pd.set_option('display.width', 220, 'display.max_columns', 40)
print('S = mean pairwise 5-label agreement over R=%d DISJOINT seed sets (higher better)'
      % R_SEED_SETS)
print('L = ZigZag transition lag, median bars, mean over seed sets (lower better)')
print('W = label switches per 100 bars (guard)   N_REG = macro legs in the FIT WINDOW')
print('=' * 190)
print(TAB.to_string(index=False))
print('=' * 190)
print('NO COMPOSITE SCORE IS COMPUTED.')

# ---- G4 must BITE. Same summarise/guard path as the real arms. -------------
_stub = [np.full(N_BARS, 'SIDEWAYS', dtype='<U8') for _ in range(R_SEED_SETS)]
_r4 = summarise_arm('STUB all-SIDEWAYS', _stub, 'degeneracy stub')
assert _r4['S'] == 100.0, 'the stub should score a PERFECT S -- that is the point'
assert not _r4['guards']['G4'][0], 'G4 FAILED TO BITE on an all-SIDEWAYS labelling'
assert _r4['void'], 'the degenerate stub was not VOIDED'
print(f"\nGUARD TEST: an all-SIDEWAYS stub scores S = {_r4['S']:.2f}% -- a PERFECT score, "
      "which is exactly how S is gamed --")
print(f"            and is VOIDED by G4 ({_r4['guards']['G4'][1]}) and G1/G2. "
      'The guard is tested, not trusted.')

In [ ]:
# ===========================================================================
# THE FULL PAIRWISE S MATRICES -- one per arm. BIMODALITY IS ONLY VISIBLE HERE.
# ===========================================================================
def is_bimodal(r):
    return (r['S_spread'] >= BIMODAL_SPREAD) and (r['S_gap'] >= BIMODAL_GAP)


for r in RESULTS:
    print(f"\n{r['name']}   S = {r['S']:.2f}%   pairwise range "
          f"[{r['S_min']:.2f}, {r['S_max']:.2f}]   spread {r['S_spread']:.2f} pp   "
          f"largest gap {r['S_gap']:.2f} pp")
    print(pd.DataFrame(r['S_matrix'],
                       index=[f'set{i}' for i in range(R_SEED_SETS)],
                       columns=[f'set{i}' for i in range(R_SEED_SETS)]
                       ).round(2).to_string())
    print(f"   6 pairwise values sorted: "
          f"{[round(v, 2) for v in sorted(r['S_pairs'])]}")
    print(f"   BIMODAL = {is_bimodal(r)}   "
          f"(needs spread >= {BIMODAL_SPREAD} pp AND gap >= {BIMODAL_GAP} pp)   "
          f"across-basin pairs (< {BASIN_LOW}%): {r['n_low_pairs']}")

# ---- HEADROOM CHECK -- REQUIRED BEFORE ANY VERDICT ------------------------
print('\n' + '=' * 96)
print(f'HEADROOM CHECK (baseline, across the R={R_SEED_SETS} disjoint seed sets)')
print('=' * 96)
S_SPREAD = BASE['S_spread']
L_SPREAD = BASE['L_hi'] - BASE['L_lo']
W_SPREAD = BASE['W_hi'] - BASE['W_lo']
print(f"  S : {BASE['S']:.2f}%   pairwise span {BASE['S_min']:.2f} -> {BASE['S_max']:.2f}"
      f"   spread = {S_SPREAD:.2f} pp")
print(f"  L : {BASE['L']:.2f} bars   per-seed-set span {BASE['L_lo']:.2f} -> "
      f"{BASE['L_hi']:.2f}   spread = {L_SPREAD:.2f} bars")
print(f"  W : {BASE['W']:.2f}/100   per-seed-set span {BASE['W_lo']:.2f} -> "
      f"{BASE['W_hi']:.2f}   spread = {W_SPREAD:.2f}")
print('  RULE (pre-declared): a change smaller than the corresponding spread is NOISE.')
for r in RESULTS[1:]:
    _sn = abs(r['S'] - BASE['S']) <= S_SPREAD
    _ln = abs(r['L'] - BASE['L']) <= L_SPREAD
    print(f"  {r['name']:<22} dS {r['S'] - BASE['S']:+7.2f} pp "
          f"({'inside' if _sn else 'OUTSIDE'} the S spread)   "
          f"dL {r['L'] - BASE['L']:+6.2f} bars ({'inside' if _ln else 'OUTSIDE'} "
          'the L spread)')
print('=' * 96)

## 8. Self-grading against the pre-registered predictions

The arithmetic below is exactly the arithmetic printed in section 2, before any
number existed. Nothing is reinterpreted.

In [ ]:
# ===========================================================================
# GRADE P1..P4 AND STATE THE DIAGNOSIS VERDICT IN PLAIN WORDS.
# ===========================================================================
DAILY_ARMS = RESULTS[1:]
BEST_DAILY = max(DAILY_ARMS, key=lambda r: r['S'])
print(f"Graded on the best daily arm by S: {BEST_DAILY['name']}  "
      f"(the other daily arm is graded alongside it, not hidden).")
print('=' * 96)

GRADE = {}
for r in DAILY_ARMS:
    p1 = (r['S'] > P1_S_MIN) and (not is_bimodal(r))
    if BASE['n_low_pairs'] == 0 and r['n_low_pairs'] == 0:
        p2, p2txt = None, 'UNTESTABLE'
    else:
        p2 = r['n_low_pairs'] < BASE['n_low_pairs']
        p2txt = 'PASS' if p2 else 'FAIL'
    dL = r['L'] - BASE['L']
    p3 = not (dL < -L_SPREAD)      # a LARGE improvement would be inconsistent with P3
    p4 = r['occ_spread'] < BASE['occ_spread']
    GRADE[r['name']] = dict(P1=p1, P2=p2, P3=p3, P4=p4)
    print(f"\n{r['name']}")
    print(f"  P1  {'PASS' if p1 else 'FAIL'}   S = {r['S']:.2f}% vs required > "
          f"{P1_S_MIN:.1f}%   BIMODAL = {is_bimodal(r)} "
          f"(spread {r['S_spread']:.2f} pp, gap {r['S_gap']:.2f} pp)")
    print(f"  P2  {p2txt}   across-basin pairs (< {BASIN_LOW}%): "
          f"{r['n_low_pairs']} vs baseline {BASE['n_low_pairs']}"
          + ('   [baseline shows NO two-basin structure in this run, so there is '
             'nothing for P2 to weaken]' if p2 is None else ''))
    print(f"  P3  {'consistent' if p3 else 'INCONSISTENT'}   dL = {dL:+.2f} bars "
          f"(baseline L spread {L_SPREAD:.2f}). A worse L is EXPECTED and is NOT "
          'counted against the architecture.')
    print(f"  P4  {'PASS' if p4 else 'FAIL'}   max occupancy spread across seed sets "
          f"{r['occ_spread']:.2f} pp vs baseline {BASE['occ_spread']:.2f} pp")

g = GRADE[BEST_DAILY['name']]
print('\n' + '=' * 96)
print('THE DIAGNOSIS VERDICT')
print('=' * 96)
if g['P1'] and (g['P2'] is True):
    print('P1 and P2 both PASS -> THE SMALL-SAMPLE DIAGNOSIS IS CONFIRMED.')
    print('Fitting direction on daily bars with long history raised label agreement past')
    print(f'{P1_S_MIN:.0f}% and removed the two-basin structure. The instability was an')
    print('effective-sample-size problem, as claimed.')
elif (not g['P1']) and (g['P2'] is False):
    print('P1 and P2 BOTH FAIL -> THE SMALL-SAMPLE DIAGNOSIS IS WRONG.')
    print('Giving the direction model many more macro regimes did NOT stabilise the fit')
    print('and did NOT remove the two-basin structure. Whatever produces the instability')
    print('is not the effective sample size. This is a real result and it is reported as')
    print('a refutation -- no threshold has been moved and no prediction reinterpreted.')
elif g['P2'] is None:
    print('P2 IS UNTESTABLE IN THIS RUN: the baseline itself shows no two-basin structure')
    print('here, so there is nothing for the daily fit to remove. The diagnosis is')
    print(f"NEITHER confirmed nor refuted. P1 alone came out "
          f"{'PASS' if g['P1'] else 'FAIL'}.")
else:
    print('MIXED: P1 =', 'PASS' if g['P1'] else 'FAIL', ' P2 =',
          'PASS' if g['P2'] else 'FAIL')
    print('The diagnosis is PARTIALLY supported only. It is not confirmed, and the half')
    print('that failed is not explained away here.')
print('-' * 96)
print('GUARDS:', ' | '.join(
    f"{r['name']}: {'VOID (' + ' '.join(k for k, v in r['guards'].items() if not v[0]) + ')' if r['void'] else 'all pass'}"
    for r in RESULTS))
if TAC_SYNTH or DAILY_SYNTH:
    print('-' * 96)
    print('*** SYNTHETIC RUN. The verdict above grades the MACHINERY, not the market.  ***')
    print('*** The real P1-P4 verdict requires the Kaggle run with real yfinance data. ***')
print('=' * 96)

# ---- dominance, for completeness. Strict, no composite. -------------------
DOMINATING = [r for r in DAILY_ARMS if (not r['void']) and r['S'] > BASE['S']
              and r['L'] < BASE['L'] and r['W'] <= BASE['W']]
print()
if DOMINATING:
    for r in DOMINATING:
        print(f"DOMINATES THE BASELINE: {r['name']}  S {BASE['S']:.2f}->{r['S']:.2f}  "
              f"L {BASE['L']:.2f}->{r['L']:.2f}  W {BASE['W']:.2f}->{r['W']:.2f}")
else:
    print('NO DOMINATING CONFIGURATION (S up AND L down AND W not worse, all guards ok).')
    print('That is the EXPECTED shape here: P3 pre-declared that L would not improve, so')
    print('strict dominance was never the bar the daily arms were meant to clear. The')
    print('question this notebook asks is P1/P2, and that is answered above.')

## 9. Figures

Three, and only three — each one decides something. The pairwise-S panel is where
P1 and P2 are judged; the (L, S) frontier is where the S-for-L trade is priced;
the reference case is where a human decides whether the trade was worth it.

In [ ]:
# ===========================================================================
# FIG 1 - the pairwise S matrices, one per arm. THIS is where P1/P2 are judged.
# ===========================================================================
SAVED_PNGS = []


def save_fig(fig, fname, dpi=130):
    fig.savefig(fname, dpi=dpi, bbox_inches='tight')
    SAVED_PNGS.append(fname)
    print(f'   saved -> {fname}')


SUFFIX = ('  [SYNTHETIC - ILLUSTRATIVE ONLY]' if (TAC_SYNTH or DAILY_SYNTH)
          else '  [real data]')
_lo = min(min(r['S_pairs']) for r in RESULTS)
VMIN = max(0.0, min(55.0, _lo - 2.0))

fig, axes = plt.subplots(2, len(RESULTS), figsize=(5.4 * len(RESULTS), 8.6),
                         gridspec_kw={'height_ratios': [3, 1.4], 'hspace': 0.42,
                                      'wspace': 0.22})
for j, r in enumerate(RESULTS):
    ax = axes[0, j]
    im = ax.imshow(r['S_matrix'], cmap='RdYlGn', vmin=VMIN, vmax=100.0)
    for i in range(R_SEED_SETS):
        for k in range(R_SEED_SETS):
            ax.text(k, i, f"{r['S_matrix'][i, k]:.1f}", ha='center', va='center',
                    fontsize=9, fontweight='bold' if i != k else 'normal',
                    color='#111111')
    # Ticks on TOP: at this figure height a bottom x-axis on the heatmap collides
    # with the strip panel's title directly beneath it.
    ax.set_xticks(range(R_SEED_SETS)); ax.set_yticks(range(R_SEED_SETS))
    ax.xaxis.set_ticks_position('top')
    ax.set_xticklabels([f'set{i}' for i in range(R_SEED_SETS)], fontsize=8)
    ax.set_yticklabels([f'set{i}' for i in range(R_SEED_SETS)], fontsize=8)
    # NO COLOURBAR: every cell is annotated with its own number, so a colour key
    # decides nothing -- and at this width it overlapped the third panel.
    ax.set_title(f"{r['name']}\nS = {r['S']:.2f}%   spread {r['S_spread']:.2f} pp"
                 + ('   [VOID]' if r['void'] else ''),
                 fontsize=10, fontweight='bold', loc='left', pad=26)

    # the 6 pairwise values as a strip -- a two-basin structure is a GAP here
    axb = axes[1, j]
    v = sorted(r['S_pairs'])
    axb.scatter(v, np.zeros(len(v)), s=90, c='#1f77b4', edgecolors='black',
                linewidths=0.7, zorder=3)
    axb.axvline(P1_S_MIN, color='#2ca02c', ls='--', lw=1.2)
    axb.axvline(BASIN_LOW, color='#d62728', ls=':', lw=1.2)
    axb.set_yticks([])
    axb.set_xlim(VMIN, 101)
    axb.grid(alpha=0.25, axis='x')
    axb.set_xlabel('the 6 pairwise agreements (%)', fontsize=9)
    axb.set_title(f"BIMODAL = {is_bimodal(r)}   largest gap {r['S_gap']:.2f} pp   "
                  f"pairs < {BASIN_LOW:.0f}%: {r['n_low_pairs']}",
                  fontsize=9, loc='left',
                  color='#d62728' if is_bimodal(r) else '#2ca02c')
fig.suptitle('FIG 1 — PAIRWISE FIT-STABILITY MATRICES  (P1 and P2 are judged here)'
             + SUFFIX + f'\ncell colour: red {VMIN:.0f}% -> green 100%   '
             f'strip: green dashed = P1 threshold {P1_S_MIN:.0f}%, '
             f'red dotted = across-basin line {BASIN_LOW:.0f}%',
             fontsize=13, fontweight='bold')
fig.subplots_adjust(top=0.84)
save_fig(fig, 'fig1_pairwise_S_matrices.png')
plt.show()

In [ ]:
# ===========================================================================
# FIG 2 - the (L, S) frontier, all arms.
# ===========================================================================
fig, ax = plt.subplots(figsize=(11, 7))
_st = {'1 BASELINE (2h fit)': ('#d62728', '*', 460),
       '2 DAILY-9FEAT': ('#1f77b4', 'o', 170),
       '3 DAILY-3FEAT': ('#2ca02c', 's', 170)}
for r in RESULTS:
    c, mk, sz = _st.get(r['name'], ('#8c564b', 'D', 140))
    if r['void']:
        ax.scatter([r['L']], [r['S']], facecolors='none', edgecolors=c, marker=mk,
                   s=sz, linewidths=2.0, zorder=4, label=r['name'] + ' [VOID]')
    else:
        ax.scatter([r['L']], [r['S']], c=c, marker=mk, s=sz, zorder=4,
                   edgecolors='black', linewidths=0.7, label=r['name'])
    ax.errorbar([r['L']], [r['S']],
                xerr=[[r['L'] - r['L_lo']], [r['L_hi'] - r['L']]],
                yerr=[[r['S'] - r['S_min']], [r['S_max'] - r['S']]],
                fmt='none', ecolor=c, elinewidth=1.1, capsize=4, alpha=0.65, zorder=3)
    # annotated ABOVE-right, so the label never lands on the P1 / baseline rules
    ax.annotate(f"N_REG={r['nreg']}  W={r['W']:.1f}", (r['L'], r['S']), fontsize=8,
                xytext=(11, 10), textcoords='offset points', color='#333333')
ax.axvline(BASE['L'], color='#d62728', ls=':', lw=1.0)
ax.axhline(BASE['S'], color='#d62728', ls=':', lw=1.0)
ax.axhline(P1_S_MIN, color='#2ca02c', ls='--', lw=1.2)
ax.margins(x=0.14, y=0.10)
_x0, _x1 = ax.get_xlim()          # extra room so the right-most annotation is not clipped
ax.set_xlim(_x0, _x1 + 0.10 * (_x1 - _x0))
ax.annotate(f'P1 threshold  S = {P1_S_MIN:.0f}%', (ax.get_xlim()[1], P1_S_MIN),
            fontsize=9, va='top', ha='right', color='#2ca02c',
            xytext=(-6, -4), textcoords='offset points')
ax.set_xlabel('L — transition lag, median bars  (LOWER IS BETTER  <—)', fontsize=10)
ax.set_ylabel('S — mean pairwise label agreement %  (HIGHER IS BETTER)', fontsize=10)
ax.set_title('error bars are the FULL across-seed-set range of L and the full pairwise '
             'range of S — anything inside them is noise', fontsize=9, loc='left')
ax.grid(alpha=0.25)
ax.legend(fontsize=9, loc='best')
fig.suptitle('FIG 2 — the (L, S) frontier: does the daily fit buy stability, and at '
             'what lag cost?' + SUFFIX, fontsize=13, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig2_frontier_L_vs_S.png')
plt.show()

In [ ]:
# ===========================================================================
# FIG 3 - THE REFERENCE CASE: the May-2025 V-bottom, BASELINE vs the best daily
# arm, in the MASTER NOTEBOOK'S regime-background style: black price line,
# FULL-HEIGHT regime bands, TIGHT y-limits asserted EQUAL and NOT zero-anchored.
# ===========================================================================
REF_START, REF_END = pd.Timestamp('2025-05-05'), pd.Timestamp('2025-05-26')
_m = (DATES >= REF_START) & (DATES <= REF_END)
if _m.sum() < 6:
    _up = max([s for s in SWINGS if s[2] > 0],
              key=lambda s: (CLOSE[s[1]] - CLOSE[s[0]]) / CLOSE[s[0]])
    _a, _b = max(0, _up[0] - 12), min(N_BARS - 1, _up[1] + 12)
    _m = np.zeros(N_BARS, bool); _m[_a:_b + 1] = True
    REF_NOTE = (f'REFERENCE WINDOW {REF_START:%Y-%m-%d} -> {REF_END:%Y-%m-%d} IS NOT IN '
                'THIS DATA SPAN — substituted the largest up-swing in the series')
else:
    REF_NOTE = f'REFERENCE CASE: {REF_START:%Y-%m-%d} -> {REF_END:%Y-%m-%d}'

REF_IDX, REF_CLOSE = DATES[_m], CLOSE[_m]
_ref_move = 100 * (REF_CLOSE[-1] - REF_CLOSE.min()) / REF_CLOSE.min()


def shade_regimes_contiguous(ax, series, alpha=0.35):
    '''Bands that BUTT UP against each other -- the master's regime_blocks ends a
    band on the LAST BAR of its run, which leaves a white stripe across every
    weekend at this zoom and reads as "no regime here", which is false. The
    PAINTER is still the master's shade_bands, unmodified.'''
    vals, idx = np.asarray(series.values), series.index
    spans, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            spans.append((vals[start], idx[start], idx[i])); start = i
    spans.append((vals[start], idx[start], idx[-1]))
    shade_bands(ax, spans, alpha=alpha)


_panels = [BASE, BEST_DAILY]
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
_ylim = None
for ax, r in zip(axes, _panels):
    lab = pd.Series(LABELS[r['name']][0], index=DATES)[_m]     # seed set 0, both panels
    shade_regimes_contiguous(ax, lab, alpha=0.35)
    ax.plot(REF_IDX, REF_CLOSE, color='black', linewidth=1.8, zorder=4)
    set_price_ylim(ax, REF_CLOSE, pad=0.03, tag=f"FIG3 {r['name']}")
    if _ylim is None:
        _ylim = ax.get_ylim()
    ax.set_ylim(*_ylim)
    ax.set_xlim(REF_IDX[0], REF_IDX[-1])
    ax.set_ylabel('Nifty close', fontsize=9)
    _d = emitted_direction(lab.values)
    _shown = 0
    for i0, i1, sgn in SWINGS:
        if not _m[i0]:
            continue
        c = '#0033cc' if sgn > 0 else '#cc0033'
        t0 = DATES[i0]
        ax.axvline(t0, color=c, ls='--', lw=1.3, zorder=6)
        _off = int(np.flatnonzero(np.asarray(lab.index) == np.datetime64(t0))[0])
        _hit = np.flatnonzero(_d[_off:] == sgn)
        _y = _ylim[0] + (0.94 - 0.085 * _shown) * (_ylim[1] - _ylim[0])
        if _hit.size:
            _lag = int(_hit[0]); t1 = lab.index[_off + _lag]
            ax.annotate('', xy=(t1, _y), xytext=(t0, _y),
                        arrowprops=dict(arrowstyle='<->', color=c, lw=1.5))
            ax.plot([t1], [REF_CLOSE[_off + _lag]], marker='o', ms=9, mfc='none',
                    mec=c, mew=2.0, zorder=7)
            ax.annotate(f"swing {'UP' if sgn > 0 else 'DOWN'} — L = {_lag} bars",
                        ((t0 + (t1 - t0) / 2) if _lag else t0, _y), fontsize=9,
                        color=c, ha='center', va='bottom', fontweight='bold')
        else:
            ax.annotate(f"swing {'UP' if sgn > 0 else 'DOWN'} — NEVER matched",
                        (t0, _y), fontsize=9, color=c, ha='left', va='bottom',
                        fontweight='bold')
        _shown += 1
    _f = np.flatnonzero(_d > 0)
    ax.set_title(f"{r['name']}{'  [VOID]' if r['void'] else ''}   S={r['S']:.2f}%   "
                 f"L={r['L']:.2f} bars (whole series)   W={r['W']:.2f}   |   "
                 + (f'first BULL bar in window: {lab.index[_f[0]]:%Y-%m-%d %H:%M}'
                    if _f.size else 'NEVER labelled BULL in this window'),
                 fontsize=11, fontweight='bold', loc='left')

fig.legend(handles=[mpatches.Patch(color=REGIME_COLORS[l], alpha=0.7, label=l)
                    for l in REGIME_LABELS],
           loc='upper center', bbox_to_anchor=(0.5, 0.925), ncol=5, fontsize=9,
           frameon=False)
axes[-1].set_xlabel('date', fontsize=10)
assert axes[0].get_ylim() == axes[1].get_ylim(), 'panel y-limits differ'
assert axes[0].get_ylim()[0] > 0, 'y-axis is zero-anchored -- tight limits required'
fig.suptitle('FIG 3 — ' + REF_NOTE + f'   ({len(REF_IDX)} bars, low->close '
             f'+{_ref_move:.2f}%)'
             + ('\n[SYNTHETIC - ILLUSTRATIVE ONLY — this is NOT the real May-2025 '
                'V-bottom]' if (TAC_SYNTH or DAILY_SYNTH) else '')
             + '\nsame bars, same intensity axis, same y-limits — only the DIRECTION '
               'fit differs',
             fontsize=13, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.905))
print(f'ASSERT OK: both panels share y-limits {axes[0].get_ylim()} '
      '(tight, NOT zero-anchored)')
save_fig(fig, 'fig3_reference_case_may2025.png')
plt.show()

## 10. Verification and the config hand-off block

In [ ]:
# ===========================================================================
# FINAL VERIFICATION.
# ===========================================================================
print('=' * 90)
print('VERIFICATION')
print('=' * 90)
print(f'  label_bars calls (all guarded)   : {LABEL_CALLS}')
print(f'  HMM fits                         : {FIT_COUNT}')
print(f'  ZigZag computations              : {ZZ_CALLS}')
print(f'  arms measured                    : {len(RESULTS)}')
print(f'  seed sets (R, DISJOINT)          : {R_SEED_SETS}  {SEEDSETS}')
print(f'  BAR_DIR_WEIGHT                   : {BAR_DIR_WEIGHT} (asserted every call)')
print(f'  causality proof 1 (per-bar)      : PASS  min source->bar gap '
      f'{_gap_days.min()} day(s)')
print(f'  causality proof 2 (truncation)   : PASS  0 mismatches over '
      f'{int(_common.sum())} labels')
print(f'  G4 driven by a degenerate stub   : PASS  (S=100.00% VOIDED)')
try:
    label_bars(None)
except AssertionError as e:
    print(f'  ZigZag leakage tripwire (live)   : ARMED -> {str(e)[:56]}...')
else:
    raise AssertionError('the leakage tripwire did NOT fire after the eval phase')
print(f'  PNGs written                     : {len(SAVED_PNGS)}  {SAVED_PNGS}')
print(f'  total runtime                    : {time.time() - T_START:.1f}s')
print('=' * 90)
if TAC_SYNTH or DAILY_SYNTH:
    print('*** SYNTHETIC - ILLUSTRATIVE ONLY. No verdict above is decision-grade.  ***')
    print('*** Re-run on Kaggle with real yfinance data.                           ***')
    print('=' * 90)

In [ ]:
# ===========================================================================
# CONFIG HAND-OFF BLOCK -- pasteable Python for the winning arm.
# ===========================================================================
_win = BEST_DAILY if (GRADE[BEST_DAILY['name']]['P1']
                      and GRADE[BEST_DAILY['name']]['P2'] is True
                      and not BEST_DAILY['void']) else BASE
_warm = 'the daily-fit arm cleared P1 AND P2' if _win is not BASE else \
        'NO daily arm cleared both P1 and P2 -- the BASELINE is carried forward unchanged'
_wa = [a for a in ARMS if a['name'] == _win['name']][0]

print('# ' + '=' * 78)
print('# REGDET V1.1 -- DAILY-FIT CONFIG HAND-OFF BLOCK')
print(f'# generated {pd.Timestamp.now():%Y-%m-%d %H:%M}   data = '
      f'{"SYNTHETIC (ILLUSTRATIVE ONLY)" if (TAC_SYNTH or DAILY_SYNTH) else "REAL yfinance"}')
print(f'# WINNER: {_win["name"]}   because {_warm}')
print(f"# S={_win['S']:.2f}%  L_med={_win['L']:.2f}  L_p75={_win['L75']:.2f}  "
      f"W={_win['W']:.2f}  N_REGIMES_OBSERVED={_win['nreg']}  "
      f"rows/param={_win['rows_per_param']}")
print('# guards: ' + ' '.join(f"{k}={'PASS' if v[0] else 'FAIL'}"
                              for k, v in _win['guards'].items())
      + f"  -> {'VOID' if _win['void'] else 'ok'}")
print('#')
print('DAILY_FIT_CONFIG = dict(')
print(f'    DIRECTION_CADENCE   = {"daily" if _wa["daily"] else "2h"!r},')
print(f'    DIRECTION_FEATURES  = {list(_wa["features"])!r},')
print(f'    INTENSITY_CADENCE   = {"2h"!r},        # unchanged in every arm')
print(f'    N_STATES            = {BASE_N},')
print(f'    COVARIANCE_TYPE     = {BASE_COV!r},')
print(f'    ENSEMBLE_K          = {ENSEMBLE_K},')
print(f'    BASE_SEED           = {BASE_SEED},')
print(f'    TRAIN_FRACTION      = {TRAIN_FRACTION},')
print(f'    DAILY_FIT_START     = {str(DAILY_START.date())!r},   '
      f'# first date ^NSEI and India VIX BOTH exist')
print(f'    DAILY_FIT_BARS      = {N_FIT_D},')
print(f'    BAR_DIR_WEIGHT      = {W_FIXED},')
print(f'    CONFIRM_BARS        = {CONFIRM_BARS},')
print(f'    INTENSITY_MODE      = {INTENSITY_MODE!r},')
print(f'    DIRECTION_MODE      = {DIRECTION_MODE!r},')
print(f'    DIRECTION_EXCLUDE   = {DIRECTION_EXCLUDE!r},')
print(f'    ESCALATION_DURING_HOLD = {ESCALATION_DURING_HOLD!r},')
print(')')
print('# PROJECTION RULE (non-negotiable): the direction state applied to a 2h bar on')
print('# session d is the daily state of the last FULLY CLOSED session, d-1 or earlier.')
print('# ' + '=' * 78)

## 11. What this notebook does and does not establish

**Does.** It measures S, L and W for a 2h-fit direction model and two daily-fit
ones on the *same* bars with the *same* intensity axis, `R = 4` disjoint seed
sets; it proves the daily→2h projection causal by truncation, not by assertion;
it checks the collinearity claim instead of assuming it; and it grades itself
against predictions frozen before the run.

**Does not.** It computes no forward return, no Sharpe and no economic metric —
deliberately. It does not re-tune `BAR_DIR_WEIGHT` or touch the intensity axis. It
does not combine S, L and W into a score. And on a sandbox run it establishes
nothing about the real market: the synthetic fallback is a plumbing test.